### **生物信息学中的统计方法 - 课程期末项目 scRNA-seq分析代码**

---

__参考文章1:__ Defining the regulatory logic of breast cancer using single-cell epigenetic and transcriptome profiling

__参考文章2:__ Comprehensive single-cell sequencing reveals the tumor microenvironment and tumor-specic characteristics

__姓名:__ 高小雅 

__学号:__ 2511210766

---

__步骤:__ 

1. **准备**：一些要用的包

2. **多样本数据读取和合并**：获取四类GSM样本（IHC免疫组化表型：TNBC、ER+、HER2+、正常）的表达矩阵，在adata.obs里标注每个细胞的GSM Batch来源和IHC分型

3. **质量控制和预处理**：过滤低质量细胞，进行标准化和对数变换以减少表达分布的偏态，筛选高变基因做PCA，去除样本间的除批次效应

4. **细胞聚类和注释**：取前25个主要PC做Leiden聚类，结合差异基因和经典Markers做细胞注释，包括对上皮细胞进行inferCNV以推断各细胞恶性程度（上皮细胞分类参考）

5. **细胞组成分析**：统计不同IHC样本的分型中各细胞类型的百分比，对不同细胞占比差异进行显著性检验，筛选出肿瘤样本里TNBC组与非TNBC组占比有显著差异的细胞

6. **差异基因分析**：用Wilcoxon检验算两组的差异基因，取两组占比有明显统计学差异的一类免疫细胞做差异基因分析

7. **细胞通讯和拟时序分析**：针对这类免疫细胞做后续的CellChat和细胞拟时序轨迹推断，检查恶性细胞与免疫微环境之间的信号传递，筛选随拟时序路径显著波动的关键基因

8. **最终识别和富集分析**：筛选TNBC特异性转录因子，锁定受这些转录因子调控而上调的关键基因（导致细胞向促癌方向转变），做对应GO/KEGG富集分析并验证生物学意义

---

__用到的数据:__

| Accession ID | Description | IHC Type | Link |
| ------------- | ----------- | ----------- | ----------- | 
| GSM7789994	| Primary breast tumor sample from patient A | scRNA-seq: Triple-Negative Breast Cancer | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789994 |
| GSM7789964	| Primary breast tumor sample from patient B | scRNA-seq: Triple-Negative  Breast Cancer | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789964 |
| GSM7789976	| Primary breast tumor sample from patient C | scRNA-seq: ER+ Breast Cancer | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789976 |
| GSM7789974	| Primary breast tumor sample from patient D | scRNA-seq: ER+ Breast Cancer | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789974 |
| GSM7789982	| Primary breast tumor sample from patient E | scRNA-seq: HER2+ Breast Cancer | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789982 |
| GSM7789986	| Normal breast tissue from patient F | scRNA-seq: Normal | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789986 |
| GSM7789988	| Normal breast tissue from patient G | scRNA-seq: Normal | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7789988 |

### **01. Import Packages**

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

import GEOparse
# sc.settings.set_figure_params(dpi=75, facecolor="white")
import os
from tqdm import tqdm
import tarfile
import gzip
import shutil
import glob
import anndata
import scipy
import harmonypy
import gprofiler
import seaborn as sns
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore')
# import episcanpy as epi

import gzip
import subprocess
import muon as mu
from muon import atac as ac
import math
import gseapy as gp

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 75
plt.rcParams["figure.facecolor"] = "white"
import infercnvpy
from statannotations.Annotator import Annotator

import decoupler as dc
from scipy.stats import spearmanr

In [ ]:
import os
import h5py
from scipy import sparse
from py_monocle import (
    learn_graph,
    order_cells,
    compute_cell_states,
    regression_analysis,
    differential_expression_genes,)

import os, glob, re, pickle
from functools import partial
from collections import OrderedDict
import operator as op
from cytoolz import compose

In [ ]:
# https://github.com/bioturing-org/py-monocle/blob/main
# !git clone https://github.com/bioturing/py-monocle.git
# !python3 -m pip install py-monocle/.

### **02. Build Adata**

In [ ]:
# 也能用GEOparse.get_GEO()一次性下载 为了快速进入正题这里就直接import本地数据集（来源见上方GEO官网链接）
samples_rna = os.listdir('/Users/ekeulseuji/Downloads/RNA')
# samples_atac = os.listdir('/Users/ekeulseuji/Downloads/ATAC')
samples_rna = [f for f in samples_rna if f != '.DS_Store']

In [ ]:
# 解压缩

for basename in samples_rna:
    source_dir = os.path.join('/Users/ekeulseuji/Downloads/RNA', basename)
    unzip_dir = os.path.join('/Users/ekeulseuji/Downloads/RNA_UNZIPPED', basename)
    
    !mkdir -p "{unzip_dir}"
    
    files = os.listdir(source_dir)
    files = [f for f in files if f != '.DS_Store']
    
    for file_name in files:
        file_path = os.path.join(source_dir, file_name)
        
        if file_name.endswith(".tar"):
            print(f"Unzipping tar: {file_name}")
            !tar -xvf "{file_path}" -C "{unzip_dir}"
        
        elif file_name.endswith(".gz"):
            print(f"Unzipping gz: {file_name}")
            output_file = os.path.join(unzip_dir, file_name[:-3])
            !gunzip -c "{file_path}" > "{output_file}"

In [ ]:
samples_rna

In [ ]:
adatas_rna = []

type_dict = {'GSM7789985':'N', 'GSM7789987':'N', 'GSM7789989':'N', 'GSM7789991':'N',
               'GSM7789963':'TN', 'GSM7789993':'TN', 'GSM7789965':'ER', 'GSM7789967':'HER2-ER',
               'GSM7789969':'HER2-ER', 'GSM7789971':'HER2-ER', 'GSM7789975':'ER', 'GSM7789981':'HER2-ER', 
               'GSM7789973':'ER', 'GSM7789977':'ER', 'GSM7789979':'ER', 'GSM7789983':'HER2-ER',
               'GSM7789986':'N', 'GSM7789988':'N', 'GSM7789990':'N', 'GSM7789992':'N',
               'GSM7789964':'TN', 'GSM7789994':'TN', 'GSM7789966':'ER', 'GSM7789968':'HER2-ER',
               'GSM7789970':'HER2-ER', 'GSM7789972':'HER2-ER', 'GSM7789976':'ER', 'GSM7789982':'HER2-ER', 
               'GSM7789974':'ER', 'GSM7789978':'ER', 'GSM7789980':'ER', 'GSM7789984':'HER2-ER'}

for basename in samples_rna:

    pre_sample_dir = os.path.join('/Users/ekeulseuji/Downloads/RNA_UNZIPPED', basename)

    if basename.endswith(".txt"):
        print("")
        # do nothing
    else:
        parts = os.listdir(pre_sample_dir)[1].split('_') # 'GSM7789968_Patient_08_3821AL_'
        sample_dir = pre_sample_dir+'/'+parts[0]+'_'+parts[1]+'_'+parts[2]+'_'+parts[3] 
        mtx_file = sample_dir + "_RNA_matrix.mtx"
        genes_file = sample_dir + "_RNA_features.tsv"
        barcodes_file = sample_dir + "_RNA_barcodes.tsv"
        # meta_file = sample_dir + "_metadata.csv"

        print(f"GSM File {basename} is Added")
        adata_sample = sc.AnnData(scipy.sparse.csr_matrix(scipy.io.mmread(mtx_file)).T)

        genes_df = pd.read_csv(genes_file, header=None, sep="\t")[1]
        genes_df = genes_df.to_frame()
        genes_df.columns = ["gene_id"]
        adata_sample.var_names = genes_df["gene_id"].astype(str)
        adata_sample.var_names_make_unique()

        barcodes_df = pd.read_csv(barcodes_file, header=None, sep="\t")
        # adata_sample.obs_names = [f"{bc}" for bc in barcodes_df[0].astype(str)]
        adata_sample.obs_names = [f"{basename}_{bc}" for bc in barcodes_df[0].astype(str)]

        # meta_df = pd.read_csv(meta_file)
        # meta_df.index = adata_sample.obs_names
        # adata_sample.obs = meta_df
        
        # batch info
        adata_sample.obs['batch'] = basename
        adata_sample.obs['type'] = type_dict[basename]
        # adata_sample.obs['modality'] = 'RNA'

        sc.pp.filter_genes(adata_sample, min_cells=3)
        adatas_rna.append(adata_sample)
        
print("RNA ALL DONE")

In [ ]:
# adata = anndata.concat(adatas_rna, join='outer', axis=0)
adata = anndata.concat(adatas_rna, join='outer', axis=0, index_unique='-')

In [ ]:
adata

### **03. Pre-processing and Feature Selection**

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
# adata_sample.obs_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=True, inplace=True)

sc.set_figure_params(figsize=(4, 4), dpi=80)
features = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for i, feature in enumerate(features):
    sc.pl.violin(adata, feature, jitter=0.4, ax=axes[i], show=False)
    axes[i].set_title(feature, fontsize=12)
    axes[i].set_xlabel('') 
plt.tight_layout()
plt.show()

In [ ]:
# exclude low/high gene counts, high mitocon perc
adata = adata[adata.obs.n_genes_by_counts > 200, :].copy()
adata = adata[adata.obs.n_genes_by_counts < 4000, :].copy()
adata = adata[adata.obs.pct_counts_mt < 20, :].copy() 

# exclude low expression genes
sc.pp.filter_genes(adata, min_cells=3)
adata.raw = adata

sc.set_figure_params(figsize=(4, 4), dpi=80)
features = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for i, feature in enumerate(features):
    sc.pl.violin(adata, feature, jitter=0.4, ax=axes[i], show=False)
    axes[i].set_title(feature, fontsize=12)
    axes[i].set_xlabel('') 
plt.tight_layout()
plt.show()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, 
                            min_mean=0.0125,
                            max_mean=3,
                            min_disp=0.5,
                            subset=True)      # adata = the subset of HighlyVariableGenes

# sc.pp.highly_variable_genes(adata, n_top_genes=2000)

print(f"Number of Highly Variable Genes: {adata.n_vars}")

adata.var.highly_variable.value_counts()

In [ ]:
sc.pp.scale(adata, max_value=10)

# sc.tl.pca(adata)
sc.tl.pca(adata, svd_solver='arpack')

sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(adata, color=["log1p_n_genes_by_counts", "log1p_total_counts", "log1p_total_counts_mt"])

In [ ]:
cell_counts = adata.obs.groupby('batch').size()

print("CELL COUNTS (groupby):")
print(cell_counts)

total_cells = adata.n_obs
cell_counts_pct = (cell_counts / total_cells * 100).round(2)

result_df = pd.DataFrame({
    'cell_count': cell_counts,
    'percentage': cell_counts_pct
})
print("")
print(result_df)

#### Batch Effect Removal

In [ ]:
sc.external.pp.harmony_integrate(
    adata,
    key="batch",
    basis="X_pca",
    adjusted_basis="X_pca_harmony",
    max_iter_harmony=20)

### **04. Clustering and Cell Type Annotation**

In [ ]:
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=15, n_pcs=30)

sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=5, resolution=0.4)

In [ ]:
# sc.settings.set_figure_params(figsize=(6.5, 5), dpi=80)
# sc.pl.umap(adata, color=['batch', 'leiden'], wspace=0.25, ncols=2)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.25))

sc.pl.umap(adata, color='batch', ax=axes[0], show=False, 
           title='Batch', legend_loc='right margin')

sc.pl.umap(adata, color='leiden', ax=axes[1], show=False,
           title='Leiden Clusters', legend_loc='on data', 
           legend_fontsize=15)

plt.tight_layout()
plt.show()

#### 分类参考：A single-cell and spatially resolved atlas of human breast cancers (https://doi.org/10.1038/s41588-021-00911-1)

#### **Epithelial Cells** 
##### - Cancer Epithelial Cells: Luminal A, Luminal B, Basal, HER2 Over-expressed, Cycling
##### - Normal Epithelial Cells: Myoephithelial, Luminal Progenitor, Mature Luminal

#### **Stromal Cells**
##### - Endothelial Cells: Tip-like (RGS5+/CXCL12+), Stalk-like (ACKR1+), Lymphatic
##### - Perivascular-like Mesenchymal Cells: Differentiated, Immature-like, Cycling
##### - Fibroblasts, Mesenchymal Cells: Myofibroblastic CAF, Inflammatory CAF (iCAF), Vascular CAF, Antigen-presenting CAF, Normal

#### **Immune Cells**
##### - T/NK Cells: CD4+, CD8+, NK, NKT, Type1 Interferon T (IFIT1), Treg, Cycling
##### - B/Plasma Cells: Naive B, Memory B, Plasma
##### - Myeloid Cells: Macrophages (LAM1/LAM2/M1/M2), Cycling, Monocytes(Inflammatory, Classical), DCs, Mast

In [ ]:
markergenes_dot_combined = {
    # Epithelial
    "Epithelial (Basal Cancer)": ["KRT14", "KRT6B", "ACTA2", "CAV1", "TAGLN", "CDKN2A"],
    "Epithelial (HER2E Cancer)": ["ERBB2", "GRB7", "STARD3", "PGAP3", "AREG", "CRYAB", "LCN2"],     
    "Epithelial (Luminal Cancer)": ["XBP1", "AGR2", "SCUBE2", "TFF3", "S100A1", "AGR3", "TFF1"],
    "Epithelial (Luminal Cancer)": ["CCND1", "STC2", "PIP", "TFF3"],
    "Epithelial (Basal Myoepithelial)": ["KRT5", "KRT14", "ACTA2", "CNN1", "MYLK", "DST"],    
    "Epithelial (Luminal Progenitor)": ["LALBA", "CSN3", "ELF5", "KIT", "ALDH1A3", "CLDN1"], 
    "Epithelial (Mature Luminal)": ["ESR1", "PGR", "FOXA1", "SPDEF", "MYB", "PRLR", "TFF3"],   
    # Immune (T B NK Plasma)
    "Immune (T)": ["CXCL13", "IL7R", "CD2", "CD28", "TNFSF8"],                       # "CCR8", "ITGAE", "CCR7" 
    "Immune (T)": ["LAG3", "IFNG", "GZMK", "CTSW", "CD8A", "CD8B", "GZMH"],          # "ZFP36", "CD27"
    "Immune (T)": ["CD5", "CD28", "CTLA4", "IL10RA", "ICOS"],                        # "ITGB7", "RGS1", "MCF2L2", "CCR4", "IL2RA", "SIT1", "FOXP3", "LRP2BP" 
    "Immune (NK/NKT)": ["FASLG", "GZMB", "IL2RB", "PRF1", "GNLY"],                   # "CD247", 
    "Immune (NK/NKT)": ["NKG7", "FCGR3A", "TYROBP", "LST1"],                         # "KLRB1", 
    "Immune (B)": ["CD22", "MS4A1","CD79A", "CD79B", "BLK", "PNOC"],                 # "CD19", "FCER2", "CD53"
    "Immune (B)": ["FCRL2", "SP140", "STAG3", "IGHD", "TNFRSF13B"],
    "Immune (Plasma)": ["JCHAIN", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "TNFRSF17"],   # "c-Myc", "MCM2", "PCNA", "CD79A"
    # Immune (Myeloid)
    "Immune (Macrophages)": ["EGR1", "FABP5", "CTSD", "SPP1", "LST1", "SIGLEC1"],    # "LGALS3", "CSTB", "TYMP"
    "Immune (Macrophages)": ["APOE", "TREM2", "APOC1", "C1QC", "C1QB"], 
    "Immune (Macrophages)": ["CXCL10", "CXCL11", "IL2RA"],                           # "TNFSF10", "CDC209", "LGALS3", "CSTB", "TYMP"
    "Immune (Monocytes)": ["LILRB1", "FCGR3A", "SERPINA1", "IL1B"],                  # "S100A9", "CTSS", "LGALS3", "IFITM3", "S100A8", "DUSP6"
    "Immune (DCs)": ["CD1C", "CLEC10A", "FCER1A", "CXCL12"],                         # "CADM1", "CLEC9A", "CD1C", "IRF8", "LAMP3", "PLD4", "IL3RA", "BATF3", "XCR1"
    "Immune (DCs)": ["IFITM1", "GZMB", "LILRB4", "IL3RA", "IRF4", "CD209"],          # "BTLA", "IRF7", "TCF4"
    "Immune (DCs)": ["LAMP3", "CCR7", "CD86", "IL1B", "IRF8"],                       # "IL7R", "CD40", "CD80", "BIRC3"
    "Immune (Mast)": ["ITGAX", "KIT", "CPA3", "TPSAB1", "IL1RL1"],                   # "CCR3", "CCL5", "TNF", "TNFSF4", "IL6", "ITGA4", "ITGB7", 
    # Endothelial
    "Endothelial (Tip)": ["RGS5", "CD34", "ESM1", "IGF2", "ANG2", "APLN", "CXCR4"],  # "ANG2", "APLN", "CXCR4",
    "Endothelial (Tip)": ["CXCL12", "CD34", "ESM1", "DLL4", "TSLP", "VCAM1"],        # "PDGF-B", "VEGFR2", "VEGFR3"
    "Endothelial (Stalk)": ["ACKR1", "VWF", "JAG1", "NOTCH1", "ETS1", "ETS2", "PTEN"], 
    "Endothelial (Lymphatic)": ["FLT4", "MMRN1", "PROX1", "KDR", "THBD", "LYVE1", "CCL21"], 
    # CAF
    "Mesenchymal (Normal)": ["DCN", "GSN", "LUM", "APOD", "CFD", "MGST1", "PDGFRA", "CD34"],
    "Mesenchymal (CAF)": ["FN1", "MMP11", "COL12A1", "COL1A1", "COL8A1", "COL11A1", "LOX", "MMP2", "FAP", "PDPN", "ACTA2", "POSTN"],     # "PLA2G2A", "ACTA2", "TAGLN", "FLI1", "MMP9", "HSPH1", "HLA-DRB1", "CFD", "HLA-DRA", "CD74"
    "Mesenchymal (CAF)": ["IL6", "CXCL1", "CXCL2", "CXCL3", "CXCL8", "CXCL12", "LIF", "TNFAIP6", "IL11", "CCL2", "CCL5"],                # "CD34", "IL6", "ALDH1A1", "KLF4", "PDGFRA", "FAP", "CXCL12", "FAP", "PDGFRA", "S100A9", 
    "Mesenchymal (CAF)": ["HLA-DRA1", "HLA-DPA1", "HLA-DQA1", "CD74", "SLP1", "SAA3", "FSP1", "HLA-DR", "CD74", "CD40", "PDPN",],        # "S100A4", "ITGB1", "MYLK", "CAV1", "TAGLN", 
    "Mesenchymal (CAF)": ["PDGFA", "RGS5", "ACTA2", "CXCL12", "CLIC3"],               # "VIM", "MYLK", "CD34", "ENG", "PDGFRB"
    # PVL
    "Mesenchymal (PVL)": ["MYH11", "ACTA1", "MYL9", "FGF10", "CD9", "TAGLN", "MYLK", "CNN1"],   # "PLA2G2A", "ACTA2", "TAGLN", "FLI1", "MMP9", "HSPH1", "HLA-DRB1", "CFD", "HLA-DRA", "CD74"
    "Mesenchymal (PVL)": ["RGS5", "NOTCH3", "CD36", "RHOB", "ITGA1", "ENG"],                    # "CD34", "IL6", "ALDH1A1", "KLF4", "PDGFRA", "FAP", "CXCL12", "FAP", "PDGFRA", "S100A9",  
    # Cycling
    "Cycling": ["MKI67", "TOP2A", "CCNB2", "KIAA0101", "ASPM", "CD3D", "PCNA", "HMGB2",  "TUBB", "STMN1", "PCLAF"],                      # "c-Myc", "MCM2", "PCNA"
    "Cycling": ['CKAP5', 'BUB1', 'DEPDC1', 'ERCC6L', 'CDC7', 'MIS18A', 'RAD51']}  

In [ ]:
sc.tl.dendrogram(adata, groupby='leiden')

sc.pl.dotplot(
    adata, 
    markergenes_dot_combined, 
    groupby='leiden',
    standard_scale='var', 
    colorbar_title='Mean expression\nin group', 
    size_title='Fraction of cells\nin group (%)',
    color_map='YlGnBu',
    dendrogram=True,
    figsize=(25, 4))

In [ ]:
# see each subcluster's TOP MARKERS
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

result_df = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
print("\n Each Subcluster's Top 10 Markers: \n")
print(result_df.head(10))

top_genes = []
for cluster in adata.obs['leiden'].cat.categories:
    top_genes.extend(result_df[cluster][:3].tolist())

# sc.pl.dotplot(adata, var_names=list(set(top_genes)), groupby='leiden', standard_scale='var')

In [ ]:
markergenes_cluster = {
    "T Cells": ["CXCR4", "IL7R", "CD3D"],             # "PTPRC", "TRBC2"
    "Luminal Epithelial": ["KRT18", "KRT8", "KRT19"], # "AZGP1", "CD24" "KRT7", "SAA1"
    "Pericytes": ["RGS5", "EDNRB", "COL18A1"],        # "IGFBP7", "EPAS1"
    "Fibroblasts": ["DCN", "COL1A2", "SERPINF1"],     # "CTSK", "FBLN1"
    "Endothelial": ["GNG11", "ADGRL4", "ADAMTS9"],    # "TM4SF1", "IFI27"
    "Myeloid": ["HLA-DRA", "TYROBP", "FCER1G"],       # "CD74", "CD68"
    "Basal Epithelial": ["KRT14", "KRT17", "ACTG2"],  # "KRT5", "CRYAB"
    "Plasma Cells": ["JCHAIN", "IGHA1", "IGKC"],      # "MZB1", "DERL3"
    "Mast Cells": ["TPSAB1", "TPSB2", "CPA3"],        # "HPGDS", "SRGN"
    "Cycling Cells": ["MKI67", "CCNB2", "TOP2A"],
    "Epithelial": ["EPCAM", "CDH1"]}        

In [ ]:
marker_id = markergenes_cluster.keys()
sc.set_figure_params(scanpy=True, fontsize=14)

In [ ]:
for mid in marker_id:
    print("")
    print(mid)
    sc.pl.umap(adata, color=markergenes_cluster[mid], 
               color_map='magma_r', size=30, frameon=False, ncols=3)

In [ ]:
dict_leiden = {'0':'T/NK Cells', 
               '1':'Luminal Epithelial', 
               '2':'Fibroblasts',
               '3':'Cycling Cells', 
               '4':'Luminal Epithelial', 
               '5':'Luminal Epithelial', 
               '6':'Myeloid Cells', 
               '7':'Endothelial', 
               '8':'Basal Epithelial', 
               '9':'Pericytes',
               '10':'Fibroblasts', 
               '11':'B/Plasma Cells',
               '12':'Mast Cells'}

In [ ]:
adata.obs['cluster_type'] = adata.obs['leiden'].map(dict_leiden)

print(adata.obs['cluster_type'].value_counts())

In [ ]:
# sc.settings.set_figure_params(figsize=(6.5, 5), dpi=80)
# sc.pl.umap(adata, color=['batch', 'leiden'], wspace=0.25, ncols=2)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.25))

sc.pl.umap(adata, color='cluster_type', ax=axes[0], show=False, 
           title='Batch', legend_loc='right margin')

sc.pl.umap(adata, color='leiden', ax=axes[1], show=False,
           title='Leiden Clusters', legend_loc='on data', 
           legend_fontsize=15)

plt.tight_layout()
plt.show()

In [ ]:
# recluster the subset into smaller clusters
def recluster_subset(input_adata, n_hvg, n_pcs, resolution):
    
    sc.pp.highly_variable_genes(input_adata,
                                n_top_genes=n_hvg, # batch_key?
                                batch_key='batch',
                                subset=True)

    sc.pp.scale(input_adata, max_value=10)
    sc.tl.pca(input_adata, n_comps=n_pcs)
    
    sc.external.pp.harmony_integrate(input_adata, key='batch', basis='X_pca', adjusted_basis='X_pca_harmony', max_iter_harmony=30)

    sc.pp.neighbors(input_adata, n_neighbors=15, use_rep='X_pca_harmony')
    # sc.tl.leiden(input_adata, resolution=resolution, key_added="subcluster")
    sc.tl.leiden(input_adata, flavor="igraph", n_iterations=5, resolution=0.3)

    sc.tl.umap(input_adata, min_dist=0.4) # min_dist=0.1 1.2

    return input_adata

In [ ]:
adata_raw = adata.raw.to_adata()

In [ ]:
def subtype_anno_by_signature(n_hvg, n_pcs, resolution, sub_signatures, sub_types, title_name, in_color):
    in_ad = adata_raw[adata_raw.obs["cluster_type"].isin(sub_types)].copy()

    sc.pp.filter_genes(in_ad, min_cells=1)
    
    if hasattr(in_ad.X, 'toarray'):
        X = in_ad.X.toarray()
    else:
        X = in_ad.X
    
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    if hasattr(in_ad.X, 'toarray'):
        in_ad.X.data[:] = X[X.nonzero()]
    else:
        in_ad.X = X
    
    sc.pp.normalize_total(in_ad, target_sum=1e4)
    sc.pp.log1p(in_ad)
        
    adata_sub = recluster_subset(in_ad, n_hvg, n_pcs, resolution)
    # sc.pl.umap(adata_sub, color="leiden", legend_loc="on data", title=f"{title_name} Cell Subclusters")
    
    for name, genes in sub_signatures.items():
        sc.tl.score_genes(adata_sub, gene_list=genes, score_name=name, use_raw=False)
        
    score_cols = list(sub_signatures.keys())
    adata_sub.obs["fine_type"] = adata_sub.obs[score_cols].idxmax(axis=1)
    adata_sub.obs["score_max"] = adata_sub.obs[score_cols].max(axis=1)
    
    # print(adata_sub.obs['cluster_type'].value_counts())
    
    adata_sub.obs['fine_type'] = pd.Categorical(adata_sub.obs['fine_type'],
                                                categories=list(sub_signatures.keys()), ordered=True)
    
    sc.settings.set_figure_params(figsize=(6, 4), dpi=80)
    
    sc.pl.umap(adata_sub, color=["fine_type"], # 'batch',
               legend_fontsize=11, size=35, alpha=0.85,
               title=f"{title_name} Cell Fine Types (Signature-based)",
               palette=in_color)

    all_genes = adata_sub.var_names.tolist()
    gene_list = []
    for sublist in sub_signatures.values():
        for gene in sublist[:5]:
            if gene in all_genes:
                gene_list.append(gene)

    # gene_list = [gene for sublist in myeloid_sig.values() for gene in sublist[:3]]
    sc.pl.matrixplot(adata_sub, var_names=gene_list, groupby="fine_type", dendrogram=False, figsize=(10,3), cmap="coolwarm", show=True) # (15,4)
    
    
    return adata_sub
    
    '''
    sc.pl.matrixplot(adata_sub, var_names=gene_list, groupby="epi_fine_type", dendrogram=False, 
                     figsize=(14,8), cmap="coolwarm", show=True)
                     
    sc.pl.dotplot(adata_sub, var_names=gene_list, groupby="epi_fine_type", dendrogram=False,
                  figsize=(14,8), cmap="RdBu_r", standard_scale="var", show=False)
    '''
    
    # sc.pl.heatmap(adata_sub, var_names=gene_list, groupby="fine_type", standard_scale="var", dendrogram=False,
    #               figsize=(14,8), cmap="RdPu", show_gene_labels=True, show=False) # "RdBu_r"

In [ ]:
immune_sig = {"CD4+ T": ["CD4", "IL7R", "CCR7", "LTB", "ANXA1", "SELL", "TCF7", "LEF1", "DUSP4", "LTB", "RBPJ", "HSPH1", "CCL20", "CCR6", "IL7R", "RORA", "MYBL1", "PTPN13"],
              "CD8+ T": ["CD8A", "CD8B", "GZMA", "GZMB", "GZMK", "GZMH", "PRF1", "IFNG", "CST7"],
              "Treg": ["FOXP3", "IL2RA", "CTLA4", "TIGIT", "CCR4", "RTKN2", "CD5", "CD28", "CCR4", "TBC1D4", "CARD16", "CTLA4", "RTKN2"], # ["FOXP3", "IL2RA", "CTLA4", "TIGIT", "CCR8"],
              "NK": ["NKG7", "GNLY", "KLRD1", "TYROBP", "FCER1G", "XCL1", "XCL2", "NCR1"],
              "NKT": ["KLRB1", "TRDC", "FGFBP2", "CD3D", "NKG7", "CD3E"],
              "B Naive": ["CD79A", "CD79B", "MS4A1", "TCL1A", "IGHM", "IGHD", "FCER2", "YBX3"],
              "B Memory": ["CD27", "CD38", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "FCRL2", "FCRL3", "FCRL4", "CD83", "BCL6", "MIR155HG", "MYC", "HLA-DPB1", "HLA-DRA", "HLA-DPA1", "MTRNR2L8", "GPR183", "HLA-DRB1", "BCL2A1", "DUSP2", "MYC", "MIR155HG" "EGR3", "REL"],
              "Plasma": ["MZB1", "JCHAIN", "SDC1", "XBP1", "IGHG1", "IGHG4", "IGHA1", "TNFRSF17", "DERL3", "PRDM1"],
              "cDC1": ["CLEC9A", "CPVL", "DNASE1L3", "CPNE3", "IDO1", "C1orf54", "TACSTD2", "XCR1", "CLNK", "CADM1"],
              "cDC2": ["FCER1A", "CD1C", "IL1R2", "CLEC10A", "CST3", "AREG", "LYZ"], # ["CD1C", "FCER1A", "CLEC10A", "CD1E"], 
              "mDC": ["LAMP3", "CCL19", "CCL22", "FSCN1", "BIRC3", "CCR7", "CCL17", "TXN"],
              "pDC": ["GZMB", "IGKC", "PTGDS", "JCHAIN", "IRF8", "IRF7", "LGALS2"],
              "Mast": ["TPSB2", "TPSAB1", "HPGD", "CTSG", "CPA3", "IL1RL1", "CMA1"], 
              "Neutrophil": ["S100A8", "S100A9", "IFITM2", "CSF3R", "FCGR3B", "AQP9", "SMCHD1", "ELANE", "PRTN3", "CEACAM8", "OLFM4", "CD177", "MME", "CXCR1"], # "FCGR3B", "CSF3R", "CXCL8", "NAMPT", 
              "Monocytes (Classical)": ["IL1B", "S100A9", "CXCL5", "SERPINB2", "EREG", "VCAN", "THBS1"], # ["S100A8", "S100A9", "VCAN", "FCN1"],
              "Monocytes (Inflammatory)": ["LST1", "CD52", "FCGR3A", "COTL1", "FCN1", "SMIM25", "SERPINA1"], # ["CXCL1", "CXCL2", "CXCL3", "EREG", "SOD2"],
              "Macrophages (M1)": ["C3", "FCGBP", "SDS", "RGS1", "OLR1", "CXCR4", "CCL4L2", "CCL20", "IL1B", "CCL3L1", "CCL4", "G0S2", "IL23A"],
              "Macrophages (M2)": ["CXCL10", "CXCL1", "CCL3", "CXCL3", "CXCL8", "ISG15", "IFIT3", "RSAD2", "IFIT2", "IFIT1", "MX1"],
              "Macrophages (LAM1)": ["FABP5", "C1QC", "C1QB", "C1QA", "CTSD", "LIPA", "TYROBP", "FCER1G"],
              "Macrophages (LAM2)": ["APOE", "LGALS3", "SPP1", "TREM2", "CSTB", "MARCKS"]}

immune_types = ["T/NK Cells", "B/Plasma Cells", "Myeloid Cells", "Mast Cells"]
immune_color = ['#2e5e9e', '#98d8fa', '#ff9c6a', '#DBCDFA', '#9e81db', '#D3EDA8', '#739d61', '#f05e46', '#A4C9E0', '#60a0cc', '#416891', '#E95351', '#A89C92', '#ED95B9', '#BDA9E8', '#7961AD', '#91C79A', '#4EA660', '#FCBC10', '#cc7218'] # '#fff9a1', '#447A91'
immune_color = 'Set2'
immune_adata = subtype_anno_by_signature(n_hvg=3000, n_pcs=35, resolution=0.6, sub_signatures=immune_sig, sub_types=immune_types, title_name="Immune Cells'", in_color=immune_color)

In [ ]:
adata.obs['cell_type'] = adata.obs['cluster_type'] 

new_annotations = immune_adata.obs['fine_type']

adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
adata.obs.update(pd.Series(new_annotations, name='cell_type'))

adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [ ]:
stromal_sig = {"Normal Fibroblasts": ["DCN", "GSN", "LUM", "APOD", "CFD", "MGST1", "PDGFRA", "CD34"],
               "Myofibroblastic CAF (myCAF)": ["FN1", "MMP11", "COL12A1", "COL1A1", "COL8A1", "COL11A1", "LOX", "MMP2", "FAP", "PDPN", "ACTA2", "POSTN", "COL1A1", "TAGLN", "ACTA2", "TAGLN", "THY1", "MYLK", "TPM1", "TPM2", "COL1A1", "COL1A2", "COL8A1", "COL12A1", "COL15A1", "TNC", "FN1", "POSTN", "TGFβ", "CTGF", "FAP", "MMP11", "LOX", "PDPN"],
               "Inflammatory CAF (iCAF)": ["IL6", "CXCL1", "CXCL2", "CXCL3", "CXCL8", "CXCL12", "LIF", "TNFAIP6", "IL11", "CCL2", "CCL5", "CCL20", "SAA1", "SAA2", "HAS1", "HAS2", "MT2A", "SOD2"],
               "Antigen-presenting CAF (apCAF)": ["HLA-DRA1", "HLA-DPA1", "HLA-DQA1", "CD74", "SLP1", "SAA3", "FSP1", "HLA-DR", "CD74", "CD40", "PDPN", "PDGFRα", "CD239", "CD321", "HLA-DRA", "CD74", "CD40"],
               "Vascular CAF (vCAF)": ["VEGF", "FGF", "PDGFA", "ANG1", "ANG2", "PDGFRβ", "RGS5", "ACTA2", "CXCL12", "CLIC3", "MMPs"],
               "Differentiated PVL (dPVL)": ["MYH11", "ACTA1", "MYL9", "FGF10", "CD9", "TAGLN", "MYLK", "CNN1"], # "TAGLN", "PDGFRB",
               "Immature-like PVL (imPVL)": ["RGS5", "NOTCH3", "CD36", "RHOB", "ITGA1", "ENG"],
               "Endothelial Tip (RGS5)": ["RGS5", "CD34", "ESM1", "IGF2", "ANG2", "APLN", "CXCR4", "DLL4", "CD34", "SOX17", "GJA5", "EFNB2"],
               "Endothelial Tip (CXCL12)": ["CXCL12", "CD34", "ESM1", "DLL4", "TSLP", "VCAM1", "PDGF-B", "VEGFR2", "VEGFR3", "DLL4", "CD34", "SOX17", "GJA5", "EFNB2"],
               "Endothelial Stalk (ACKR1)": ["ACKR1", "VWF", "JAG1", "NOTCH1", "ETS1", "ETS2", "PTEN", "VWF", "CD36", "NR2F2", "EMCN", "EPHB4"]}

stromal_types = ["Fibroblasts", "Perivascular-like", "Endothelial"]
# stromal_color = ['#98d6a5', "#f2cb4b", '#578f4c', "#f77457", "#f79f57", '#dbc6a7', '#877863', '#696969', '#505050', '#363636', '#1A1A1A', '#BAAB79', '#ff9c6a', '#FEE00C'] # '#368650', '#598251',
stromal_color = "Set2"
stromal_adata = subtype_anno_by_signature(n_hvg=3500, n_pcs=35, resolution=0.6, sub_signatures=stromal_sig, sub_types=stromal_types, title_name="Stromal Cells'", in_color=stromal_color)

In [ ]:
new_annotations = stromal_adata.obs['fine_type']

adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
adata.obs.update(pd.Series(new_annotations, name='cell_type'))

adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

#### **04.inferCNV**

In [ ]:
adata_raw = adata.raw.to_adata()

In [ ]:
epi_types = ['Luminal Epithelial', 'Basal Epithelial']

In [ ]:
max_cells_per_type = 1000000  # one sample at a time so no restrict at all
# adata_raw.obs_names_make_unique()
all_indices = []
obs_all = adata_raw.obs[adata_raw.obs['cluster_type'].isin(immune_types)]

for SampleType in immune_types:
    type_subset = obs_all[obs_all['cluster_type'] == SampleType]
    if len(type_subset) > max_cells_per_type:
        sampled_indices = type_subset.sample(n=max_cells_per_type, random_state=42).index
        all_indices.extend(sampled_indices)
    else:
        all_indices.extend(type_subset.index)
        
max_epi = 5000 
epithelial_indices = adata_raw.obs[adata_raw.obs["cluster_type"].isin(epi_types)].index # .isin(epi_types)].index
if len(epithelial_indices) > max_epi:
    epithelial_indices = pd.Series(epithelial_indices).sample(n=max_epi, random_state=42).tolist()
    
total_indices = list(epithelial_indices) + list(all_indices)

adata_cnv1 = adata_raw[total_indices].copy()

reference_cells = [c for c in all_indices if c in adata_cnv1.obs_names]
norm_cell_names_str = ",".join(reference_cells)

print(f"n cells: {adata_cnv1.n_obs}")
print(f"n genes: {adata_cnv1.n_vars}")

In [ ]:
# https://github.com/broadinstitute/infercnv/blob/master/scripts/gtf_to_position_file.py
# cd ~/Downloads
# curl -O https://cf.10xgenomics.com/supp/cell-exp/refdata-gex-GRCh38-2020-A.tar.gz
# tar -xzf refdata-gex-GRCh38-2020-A.tar.gz
# python3 ~/Downloads/gtf_to_position_file.py \
#  --attribute_name gene_name \
#  ~/Downloads/refdata-gex-GRCh38-2020-A/genes/genes.gtf \
#  gene_position.txt

gene_pos_txt = "/Users/ekeulseuji/Downloads/gene_position.txt"

gene_pos = pd.read_csv(gene_pos_txt, sep='\t', names=['gene', 'chr', 'start', 'end'])
gene_pos.set_index('gene', inplace=True)

adata_cnv1.var = adata_cnv1.var.merge(gene_pos, left_index=True, right_index=True, how='left')

adata_cnv1 = adata_cnv1[:, ~adata_cnv1.var['chr'].isna()].copy()

print(adata_cnv1.n_vars) # 坐标匹配后的基因数
print('')
print(adata_cnv1.var[['chr', 'start', 'end']].head())

In [ ]:
# ValueError: Genomic positions not found. There need to be `chromosome`, `start`, and `end` columns in `adata.var`. 
adata_cnv1.var = adata_cnv1.var.rename(columns={"chr": "chromosome"})

infercnvpy.tl.infercnv(
    adata_cnv1,
    reference_key="cluster_type",
    reference_cat=immune_types,
    window_size=100, 
    step=10,
    inplace=True)

In [ ]:
adata_cnv1.var.loc[:, ["chromosome", "start", "end"]].head()

In [ ]:
infercnvpy.tl.pca(adata_cnv1)
infercnvpy.pp.neighbors(adata_cnv1)
infercnvpy.tl.leiden(adata_cnv1)

In [ ]:
infercnvpy.tl.umap(adata_cnv1)
infercnvpy.tl.cnv_score(adata_cnv1)

sc.pl.embedding(adata_cnv1, basis="cnv_umap", color=["cnv_score", "cluster_type"])

In [ ]:
bins = [0, 0.005, 0.010, 0.015, 0.020, np.inf]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

adata_cnv1.obs['cnv_status_quantitative'] = pd.cut(adata_cnv1.obs['cnv_score'], bins=bins, labels=labels, right=False)

In [ ]:
# fig, ((ax1, ax2, ax3)) = plt.subplots(1, 3, figsize=(9, 1), gridspec_kw={"wspace": 0.5})
# ax4.axis("off")
# fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# sc.pl.umap(adata_cnv1, color="cnv_leiden", ax=axes[0], show=False)
# sc.pl.umap(adata_cnv1, color="cnv_score", ax=axes[1], show=False)
# sc.pl.umap(adata_cnv1, color="CellType", ax=axes[2], show=False)

# plt.tight_layout()
# plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"wspace": 0.5})

sc.pl.umap(adata_cnv1, color="cnv_score", s=30, ax=ax1, show=False)
sc.pl.umap(adata_cnv1, color='cnv_status_quantitative', palette=['#4EA660', '#91C79A', '#ffda8e', '#FFB77D', '#E95351'], s=30, ax=ax2)
# sc.pl.umap(adata_cnv1, color="cluster_type", s=30, palette=['#e09358', '#e07858', '#e05858', '#bf3737', '#D3EDA8', '#91C79A', '#4EA660', '#BDA9E8', '#DBCDFA', '#9586C2', '#F5EEB0', '#D9BB82', '#A4C9E0', '#6FA7BF', '#B5E6F5'], ax=ax2)

In [ ]:
# print(adata_cnv1.obs['cnv_status_quantitative'].value_counts())

# sc.pl.umap(adata_cnv1, color=['cnv_score', 'cnv_status_quantitative'], cmap='RdYlBu_r')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"wspace": 0.5})
# infercnvpy.pl.umap(adata_cnv1, color="cnv_leiden", s=30, ax=ax1, legend_loc="on data", show=False)
# infercnvpy.pl.umap(adata_cnv1, color='cnv_status_quantitative', s=30, ax=ax2)

infercnvpy.pl.umap(adata_cnv1, color="cluster_type", s=30, ax=ax1, palette='Set2', show=False)
infercnvpy.pl.umap(adata_cnv1, color='cnv_status_quantitative', palette=['#4EA660', '#91C79A', '#ffda8e', '#FFB77D', '#E95351'], s=30, ax=ax2)

In [ ]:
df = adata_cnv1.obs[['cluster_type', 'cnv_status_quantitative']].copy()

# Cross Tabulation
counts = pd.crosstab(df['cluster_type'], df['cnv_status_quantitative'])

# every col added up to 100%
percent = counts.div(counts.sum(axis=1), axis=0) * 100

# percent = percent.sort_values(by='High', ascending=False)
percent['High_VeryHigh'] = percent['Very High'] # percent['High'] + percent['Very High'] + percent['Medium']
percent = percent.sort_values(by='High_VeryHigh', ascending=True)
percent = percent.drop('High_VeryHigh', axis=1)

color_dict2 = {
    'Very High': '#E95351',
    'High': '#f4a261',
    'Medium': '#ffda8e',
    'Low': '#91C79A',
    'Very Low': '#4EA660'}

ax = percent.plot(
    kind='barh',
    stacked=True, 
    figsize=(10, 5),
    color=[color_dict2.get(x, '#333333') for x in percent.columns],
    alpha=1, grid=False)

plt.title('Proportion of inferCNV Status by Cluster Type\n', fontsize=15)
plt.xlabel('Percentage (%)', fontsize=12)
plt.ylabel('Cluster Type', fontsize=12)
plt.legend(title='inferCNV Status', bbox_to_anchor=(1.05, 1), loc='upper left')

for p in ax.patches:
    width = p.get_width()
    if width > 5:
        ax.annotate(f'{width:.1f}%', 
                    (p.get_x() + width/2, p.get_y() + p.get_height()/2),
                    ha='center', va='center', fontsize=9, color='white')

plt.tight_layout()
plt.show()

In [ ]:
ref_obs = adata_cnv1.obs[adata_cnv1.obs['cluster_type'].isin(immune_types)]

ref_median = ref_obs["cnv_score"].median()
# cnv_threshold = ref_median 

ref_mean = ref_obs["cnv_score"].mean()
ref_std = ref_obs["cnv_score"].std()

cnv_threshold_low = ref_median - 1.64*ref_std
cnv_threshold_high = ref_median + 1.64*ref_std

In [ ]:
sc.settings.set_figure_params(figsize=(10, 5), dpi=80)

# sns.kdeplot(data=adata_cnv1.obs, x="cnv_score", hue="cell_type")
# plt.axvline(cnv_threshold_high, color='tomato', linestyle='--')
# plt.axvline(cnv_threshold_low, color='cadetblue', linestyle='--')

obs_epi = adata_cnv1.obs[adata_cnv1.obs["cluster_type"].isin(epi_types)]
obs_ref = adata_cnv1.obs[adata_cnv1.obs["cluster_type"].isin(immune_types)]

sns.kdeplot(
    data=obs_epi, x="cnv_score", hue="cluster_type", 
    alpha=1, fill=False, linewidth=1.5, common_norm=False)

sns.kdeplot(data=obs_ref, x="cnv_score", hue="cluster_type", alpha=0.4, fill=False, linestyle='--', linewidth=1.5, common_norm=False, legend=False)


plt.axvline(cnv_threshold_high, color='tomato', linestyle='--', label=f'UpperThreshold: {cnv_threshold_high:.4f}')
plt.axvline(cnv_threshold_low, color='cadetblue', linestyle='--', label=f'LowerThreshold: {cnv_threshold_low:.4f}')

handles, labels = plt.gca().get_legend_handles_labels()

n_epi = len(obs_epi["cluster_type"].unique())
plt.legend(handles[:n_epi], labels[:n_epi] + labels[-2:]) # title="Cell Types", # labels[:n_epi] + labels[-2:],

plt.title("CNV Score Distribution of Epithelial (Solid Lines) & Reference (Dashed Lines) Cells")
plt.show()

In [ ]:
sub_adata = adata_cnv1[adata_cnv1.obs["cluster_type"].isin(epi_types)].copy()

sc.pp.scale(sub_adata, max_value=10)
sc.tl.pca(sub_adata, n_comps=25)

sc.external.pp.harmony_integrate(sub_adata, key='batch', basis='X_pca', adjusted_basis='X_pca_harmony', max_iter_harmony=20)

sc.pp.neighbors(sub_adata, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.leiden(sub_adata, resolution=0.4, key_added="subcluster")

sc.tl.umap(sub_adata, min_dist=2) # min_dist=0.1

sc.pl.umap(sub_adata, color="subcluster", legend_loc="on data", title="Epithelial Cell Subclusters")

sub_adata.obs['cnv_malignant_status'] = 'Unassigned'

cnv_scores = sub_adata.obs['cnv_score'].values
mask_has_score = ~pd.isna(cnv_scores)

malignant_status = np.where(
    mask_has_score,
    np.where(cnv_scores >= cnv_threshold_high, 'Malignant', 'Non-malignant'),
    'unassigned'
)

sub_adata.obs['cnv_malignant_status'] = malignant_status

print(sub_adata.obs['cnv_malignant_status'].value_counts())

In [ ]:
print(sub_adata.obs['type'].value_counts())

In [ ]:
IHC_type_map = {
    'ER': 'ER+',
    'mER': 'ER+',
    'B1': 'Normal',
    'N': 'Normal',
    'N-NF': 'Normal',
    'N-NE': 'Normal',
    'HER2': 'HER2+',
    'HER2-ER': 'HER2+',
    'TN': 'TNBC',
    'TN-B1': 'TNBC'}

sub_adata.obs['IHC_type'] = sub_adata.obs['type'].astype(str).map(IHC_type_map).astype('category')

print(sub_adata.obs['IHC_type'].value_counts())

if sub_adata.obs['IHC_type'].isna().any():
    print("error: non-mapped")

adata.obs['IHC_type'] = adata.obs['type'].astype(str).map(IHC_type_map).astype('category')

print(adata.obs['IHC_type'].value_counts())

if adata.obs['IHC_type'].isna().any():
    print("error: non-mapped")

In [ ]:
IHC_adata = sub_adata[sub_adata.obs["IHC_type"].isin(['ER+', 'HER2+', 'TNBC'])].copy()

mali_adata = IHC_adata[IHC_adata.obs['cnv_malignant_status'].isin(['Malignant'])].copy()

# infercnvpy.pl.umap(mali_adata, color="IHC_type", title="IHC Phenotypes of Epithelial Cells with Malignant inferCNV Status")

epi_cancer_signatures = {
    "Basal Cancer Cells": ["EMP1", "TAGLN", "TTYH1", "RTN4", "TK1", "BUB3", "IGLV3.25", "FAM3C", "TMEM123", "KDM5B", "KRT14", "ALG3", "KLK6", "EEF2", "NSMCE4A", "LYST", 
                 "DEDD", "HLA.DRA", "PAPOLA", "SOX4", "ACTR3B", "EIF3D", "CACYBP", "RARRES1", "STRA13", "MFGE8", "FRZB", "SDHD", "UCHL1", "TMEM176A", "CAV2", "MARCO", 
                 "P4HB", "CHI3L2", "APOE", "ATP1B1", "C6orf15", "KRT6B", "TAF1D", "ACTA2", "LY6D", "SAA2", "CYP27A1", "DLK1", "IGKV1.5", "CENPW", "RAB18", "TNFRSF11B", 
                 "VPS28", "HULC", "KRT16", "CDKN2A", "AHNAK2", "SEC22B", "CDC42EP1", "HMGA1", "CAV1", "BAMBI", "TOMM22", "ATP6V0E2", "MTCH2", "PRSS21", "HDAC2", "ZG16B", 
                 "GAL", "SCGB1D2", "S100A2", "GSPT1", "ARPC1B", "NIT1", "NEAT1", "DSC2", "RP1.60O19.1", "MAL2", "TMEM176B", "CYP1B1", "EIF3L", "FKBP4", "WFDC2", "SAA1", 
                 "CXCL17", "PFDN2", "UCP2", "RAB11B", "FDCSP", "HLA.DPB1", "PCSK1N", "C4orf48", "CTSC"],
    "HER2E Cancer Cells": ["PSMA2", "PPP1R1B", "SYNGR2", "CNPY2", "LGALS7B", "CYBA", "FTH1", "MSL1", "IGKV3.15", "STARD3", "HPD", "HMGCS2", "ID3", "NDUFB8", "COTL1", "AIM1", "MED24", "CEACAM6", "FABP7", "CRABP2",
                 "NR4A2", "COX14", "ACADM", "PKM", "ECH1", "C17orf89", "NGRN", "ATG5", "SNHG25", "ETFB", "EGLN3", "CSNK2B", "RHOC", "PSENEN", "CDK12", "ATP5I", "ENTHD2", "QRSL1", "S100A7", "TPM1",
                 "ATP5C1", "HIST1H1E", "LGALS1", "GRB7", "AQP3", "ALDH2", "EIF3E", "ERBB2", "LCN2", "SLC38A10", "TXN", "DBI", "RP11.206M11.7", "TUBB", "CRYAB", "CD9", "PDSS2", "XIST", "MED1", "C6orf203",
                 "PSMD3", "TMC5", "UQCRQ", "EFHD1", "BCAM", "GPX1", "EPHX1", "AREG", "CDK2AP2", "SPINK8", "PGAP3", "NFIC", "THRSP", "LDHB", "MT1X", "HIST1H4C", "LRRC26", "SLC16A3", "BACE2", "MIEN1",
                 "AR", "CRIP2", "NME1", "DEGS2", "CASC3", "FOLR1", "SIVA1", "SLC25A39", "IGHG1", "ORMDL3", "KRT81", "SCGB2B2", "LINC01285", "CXCL8", "KRT15", "RSU1", "ZFP36L2", "DKK1", "TMED10", "IRX3","S100A9", "YWHAZ"],
    "LuminalA Cancer Cells": ["SH3BGRL", "HSPB1", "PHGR1", "SOX9", "CEBPD", "CITED2", "TM4SF1", "S100P", "KCNK6", "AGR3", "MPC2", "CXCL13", "RNASET2", "DDIT4", "SCUBE2", "KRT8", 
                    "MZT2B", "IFI6", "RPS26", "TAGLN2", "SPTSSA", "ZFP36L1", "MGP", "KDELR2", "PPDPF", "AZGP1", "AP000769.1", "MYBPC1", "S100A1", "TFPI2", "JUN", 
                    "SLC25A6", "HSP90AB1", "ARF5", "PMAIP1", "TNFRSF12A", "FXYD3", "RASD1", "PYCARD", "PYDC1", "PHLDA2", "BZW2", "HOXA9", "XBP1", "AGR2", "HSP90AA1"],
    "LuminalB Cancer Cells": ["UGCG", "ARMT1", "ISOC1", "GDF15", "ZFP36", "PSMC5", "DDX5", "TMEM150C", "NBEAL1", "CLEC3A", "GADD45G", "MARCKS", "FHL2", "CCDC117", "LY6E", "GJA1", 
                    "PSAP", "TAF7", "PIP", "HSPA2", "DSCAM.AS1", "PSMB7", "STARD10", "ATF3", "WBP11", "MALAT1", "C6orf48", "HLA.DRB1", "HIST1H2BD", "CCND1", "STC2", 
                    "NR4A1", "NPY1R", "FOS", "ZFAND2A", "CFL1", "RHOB", "LMNA", "SLC40A1", "CYB5A", "SRSF5", "SEC61G", "CTSD", "DNAJC12", "IFITM1", "MAGED2", "RBP1", 
                    "TFF1", "APLP2", "TFF3", "TRH", "NUPR1", "EMC3", "TXNIP", "ARPC4", "KCNE4", "ANPEP", "MGST1", "TOB1", "ADIRF", "TUBA1B", "MYEOV2", "MLLT4", "DHRS2", "IFITM2"]}

for name, genes in epi_cancer_signatures.items():
    sc.tl.score_genes(mali_adata, gene_list=genes, score_name=name, use_raw=False)
        
score_cols = list(epi_cancer_signatures.keys())
mali_adata.obs["fine_type"] = mali_adata.obs[score_cols].idxmax(axis=1)
mali_adata.obs["score_max"] = mali_adata.obs[score_cols].max(axis=1)
    
print(mali_adata.obs["fine_type"].value_counts())

mali_adata.obs['fine_type'] = pd.Categorical(mali_adata.obs['fine_type'], categories=list(epi_cancer_signatures.keys()), ordered=True)
    
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"wspace": 0.5})
    
infercnvpy.pl.umap(mali_adata, color=["fine_type"], legend_fontsize=11, size=45, ax=ax1, alpha=0.6, title=f"Malignant Epithelial Cell's Fine Types (Signature-based)", palette=['#aa1a7d', '#6baed5', '#f6be2a', '#c96f2e'], show=False)

sc.pl.umap(mali_adata, color='fine_type', size=45, ax=ax2, title="Malignant Epithelial Cell Subclusters")

In [ ]:
epi_normal_signatures = {
    "Basal Myoepithelial Cells": ["KRT5", "KRT14", "KRT17", "OXTR", "ACTA2", "TAGLN", "CNN1", "MYLK", "DST", "MYL9"],
    "Luminal Progenitor Cells": ["LALBA", "SLC34A2", "CSN3", "CYP24A1", "PIGR", "KIT", "ELF5", "NCALD", "S100A8", "FOLR1", "SLC28A3", "LBP", "ATP6V1B1", 
                           "PDZK1IP1", "SORBS2", "CLDN1", "GJB2", "RASGEF1C", "WFDC3", "IL4I1", "ANPEP", "C3", "FOXI1", "TSPAN33", "XDH", "TNFAIP2",
                           "BBOX1", "MFI2", "SLC13A2", "RASAL1", "C1QTNF1", "CD14", "HAPLN3", "ACCN2", "PLB1", "GALNTL2", "ATP6V1C2", "ALDH1A3", 
                           "SECTM1", "QPCT", "ACSL1", "C10orf90", "RPS6KL1", "CTSC", "HIVEP3", "RFTN2", "CKMT1B", "GGT5", "IL15", "CSN2", "CCDC88B",
                           "NOXO1", "DAPP1", "GNE", "ITPR2", "HSD17B12", "LPCAT1"],
    "Mature Luminal Cells": ["FOXA1", "DNAJC12", "MLPH", "SPINK1", "RASEF", "BATF", "TOX3", "HMGCS2", "EEF1A2", "CITED1", "ABCC8", "PRLR", 
                       "SLC7A2", "SLC16A5", "ALDH3B2", "SLC40A1", "WNT4", "SPDEF", "REEP6", "SULT2B1", "PGR", "PVALB", "MBOAT1", "TNFSF11",
                       "TGM2", "FLVCR2", "FAAH", "SLC22A18", "ESR1", "MYB", "WNK4", "FGL1", "C17orf28", "ALCAM", "KIAA1244", "TSPAN1", "FER1L4",
                       "WNT5A", "TBX3", "PLEKHG3", "SORT1", "C1orf210", "CCDC92", "SLC44A4", "SPRR1A", "ATP6V0E2", "GALE", "TRIM6", "TSPAN13", "KBTBD4", 
                       "PROM2", "C14orf45", "WNT7B", "PSD4", "CASZ1", "DUSP10", "PVRL4", "PHKA1", "GPRC5C", "PON3", "HDAC11", "HIST1H4K", "KLHL5",
                       "TMEM8A", "PTPN6", "FGF13", "GADD45G", "C11orf35", "SLC7A4", "IL13RA1", "FYCO1", "ZSCAN18", "RABL3", "MEIS3", "FAM63A", "HES6",
                       "CACNB3", "G6PD", "ALDH3B1", "EPS8L1", "LNX2", "TMCO3", "C19orf51", "TP53INP2", "FBXO36", "C22orf25", "SGMS1", "HOXB2", "ERN1",
                       "HIPK2", "ANKMY2", "ZFHX2", "LAMA5", "SLC25A23", "HSD11B2", "YIPF6", "AQP11", "LRRC48", "GMPR", "HIST1H4J", "ACPL2", "CACNG4",
                       "ACOT11", "MEIS1", "LPIN2", "TMPRSS6", "SCMH1", "PAK4", "TUBG1", "VOPP1", "ABCA7", "ZDHHC1", "WHSC1L1", "EDEM1", "BTRC", "VPS33B"]}

In [ ]:
IHC_N_adata = sub_adata[sub_adata.obs["IHC_type"].isin(['Normal'])].copy()

non_mali_adata = IHC_adata[IHC_adata.obs['cnv_malignant_status'].isin(['Non-malignant'])].copy()

ihc_normal_cells = IHC_N_adata.obs_names
non_malignant_cells = non_mali_adata.obs_names

target_cells = ihc_normal_cells.union(non_malignant_cells)

norm_adata = sub_adata[target_cells].copy()

for name, genes in epi_normal_signatures.items():
    sc.tl.score_genes(norm_adata, gene_list=genes, score_name=name, use_raw=False)
        
score_cols = list(epi_normal_signatures.keys())
norm_adata.obs["fine_type"] = norm_adata.obs[score_cols].idxmax(axis=1)
norm_adata.obs["score_max"] = norm_adata.obs[score_cols].max(axis=1)
    
print(norm_adata.obs["fine_type"].value_counts())

norm_adata.obs['fine_type'] = pd.Categorical(norm_adata.obs['fine_type'], categories=list(epi_normal_signatures.keys()), ordered=True)
    
sc.settings.set_figure_params(figsize=(8, 5), dpi=80)
    
# infercnvpy.pl.umap(norm_adata, color=["fine_type"], legend_fontsize=11, size=18, alpha=0.6, title=f"Non-Malignant Epithelial Cell's Fine Types (Signature-based)", palette=['#BDA9E8', '#a2d59b', '#148843'])

norm_adata.obs['fine_type'] = pd.Categorical(norm_adata.obs['fine_type'], categories=list(epi_normal_signatures.keys()), ordered=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"wspace": 0.5})
    
infercnvpy.pl.umap(norm_adata, color=["fine_type"], legend_fontsize=11, size=45, ax=ax1, alpha=0.6, title=f"Non-Malignant Epithelial Cell's Fine Types (Signature-based)", palette=['#BDA9E8', '#a2d59b', '#148843'], show=False)

sc.pl.umap(norm_adata, color='fine_type', size=45, ax=ax2, title="Non-Malignant Epithelial Cell Subclusters")

In [ ]:
new_annotations = mali_adata.obs['fine_type']

adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
adata.obs.update(pd.Series(new_annotations, name='cell_type'))
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

sub_adata.obs['cell_type'] = sub_adata.obs['cell_type'].astype(str)
sub_adata.obs.update(pd.Series(new_annotations, name='cell_type'))
sub_adata.obs['cell_type'] = sub_adata.obs['cell_type'].astype('category')

new_annotations = norm_adata.obs['fine_type']

adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
adata.obs.update(pd.Series(new_annotations, name='cell_type'))
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

sub_adata.obs['cell_type'] = sub_adata.obs['cell_type'].astype(str)
sub_adata.obs.update(pd.Series(new_annotations, name='cell_type'))
sub_adata.obs['cell_type'] = sub_adata.obs['cell_type'].astype('category')

In [ ]:
infercnvpy.pl.umap(sub_adata, color=['cell_type'], legend_fontsize=11, size=45, alpha=0.6, title=f"Epithelial Cells' Finer Cell Type Annotations",
                  palette=['#aa1a7d', '#BDA9E8', '#6baed5', '#a2d59b', '#f6be2a', '#c96f2e', '#148843'])

In [ ]:
epi_types = [
    'LuminalA Cancer Cells', 'LuminalB Cancer Cells', 'HER2E Cancer Cells', 'Basal Cancer Cells',
    'Basal Myoepithelial Cells', 'Mature Luminal Cells', 'Luminal Progenitor Cells']

epi_adata = adata[adata.obs["cell_type"].isin(epi_types)].copy()

all_sigs = {**epi_cancer_signatures, **epi_normal_signatures}

gene_list = []
for sublist in all_sigs.values():
    for gene in sublist[:70]:
        if gene in epi_adata.var_names:
            if gene not in gene_list:
                gene_list.append(gene)

# gene_list = [gene for sublist in myeloid_sig.values() for gene in sublist[:3]]

sc.pl.matrixplot(epi_adata, var_names=gene_list, groupby="cell_type", dendrogram=False, 
                     figsize=(24,2), cmap="coolwarm", show=True)

In [ ]:
colors = ['#941456', '#7CBB5F', '#A56BA7', '#D3EDA8', '#E0A7C8', '#E069A6', '#9DC3C3']
epi_adata.uns['cell_type_colors'] = colors

sc.tl.rank_genes_groups(epi_adata, 'cell_type', method='wilcoxon')
top_genes = pd.DataFrame(epi_adata.uns['rank_genes_groups']['names']).head(15).melt()['value'].unique()

'''
sc.pl.rank_genes_groups_heatmap(epi_adata, n_genes=15, groupby='cell_type', standard_scale='var', 
                                cmap='coolwarm', show_gene_labels=True, swap_axes=False, dendrogram=True)
'''

sc.pl.heatmap(epi_adata, var_names=top_genes, 
    groupby='cell_type', 
    cmap='coolwarm',
    standard_scale='var',
    figsize=(18, 8),
    dendrogram=True,
    show_gene_labels=True,
    swap_axes=False,)

In [ ]:
sc.settings.set_figure_params(figsize=(14, 7), dpi=80)

sc.pl.umap(
    adata, 
    color='cell_type', 
    size=6, 
    title='Finer Cell Types',
    legend_loc='on data',
    legend_fontsize=35,
    legend_fontoutline=2,
    frameon=True,
    show=False)

plt.legend([], frameon=False)
plt.show()

### **05. Cell Composition Analysis**

In [ ]:
sample_color_dict = {
    'Luminal Epithelial': '#9f9ac4',
    'Fibroblasts': '#6baed5',
    'T/NK Cells': '#9bc9dd',
    'B/Plasma Cells': '#02779b',
    'Myeloid Cells': '#3bab5a',
    'Pericytes': '#a2d59b',
    'Endothelial': '#ad9f93',
    'Basal Epithelial': '#f6be2a',
    'Mast Cells': '#f08c44',
    'Cycling Cells': '#c76b75'}

tmp = adata.obs.groupby(['IHC_type', 'cluster_type']).size().unstack(fill_value=0)
tmp_pct = tmp.div(tmp.sum(axis=1), axis=0) * 100

ordered_columns = [col for col in sample_color_dict.keys() if col in tmp_pct.columns]
extra_columns = [col for col in tmp_pct.columns if col not in sample_color_dict.keys()]
if extra_columns:
    ordered_columns.extend(extra_columns)

tmp_pct_ordered = tmp_pct[ordered_columns]


fig, ax = plt.subplots(figsize=(16, 5))

bars = tmp_pct_ordered.plot(kind='barh', stacked=True, ax=ax, color=[sample_color_dict[col] for col in tmp_pct_ordered.columns], legend=False, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Proportion (%)', fontsize=12)
ax.set_ylabel('IHC Type', fontsize=12)
ax.set_title('Cell Type Composition by IHC Type', fontsize=14)

ax.grid(axis='x', alpha=0.3, linestyle='--')

for bar in ax.patches:
    width = bar.get_width()
    if width > 3: 
        ax.annotate(f'{width:.1f}%', 
                    (bar.get_x() + width/2, bar.get_y() + bar.get_height()/2),
                    ha='center', va='center', 
                    fontsize=8, color='white', 
                    fontweight='bold')

handles, labels = ax.get_legend_handles_labels()
handles = handles[::-1]
labels = labels[::-1]

legend = ax.legend(handles, labels,
                   title='Cell Types',
                   bbox_to_anchor=(1.02, 1),
                   loc='upper left',
                   frameon=True,
                   fancybox=True,
                   shadow=False,
                   fontsize=9,
                   title_fontsize=10)

plt.tight_layout()
plt.subplots_adjust(right=0.78)

plt.show()

In [ ]:
tmp_df = adata.obs[['batch', 'cluster_type']].copy()

# 算样本中各细胞类型占比
batch_totals = tmp_df.groupby('batch').size().rename('batch_total')
type_counts = tmp_df.groupby(['batch', 'cluster_type']).size().reset_index(name='count')

type_counts = type_counts.merge(batch_totals, on='batch')
type_counts['prop'] = type_counts['count'] / type_counts['batch_total']

group_composition_norm = type_counts.pivot(index='batch', columns='cluster_type', values='prop').fillna(0)
# group_composition_norm = group_composition_norm.sort_values(by='Cycling Cells', ascending=True)

fig, ax = plt.subplots(figsize=(8.25, 6)) 

group_composition_norm.plot(kind='bar', stacked=True, ax=ax, 
    color=[sample_color_dict.get(c, '#gray') for c in group_composition_norm.columns], width=0.7)


plt.title("Normalized Origin of Cell Subtypes by Samples' IHC Phenotype\n", fontsize=15)
plt.ylabel('Normalized Proportion in Samples')
plt.xlabel('Sample IDs') 
plt.xticks(rotation=45, ha='right')
plt.legend(title='Cell Types', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.grid(axis='y', linestyle='--', alpha=0.5)
ax.grid(False)
plt.tight_layout()
plt.show()

#### **Analyze the Cell Abundance in Different IHC Sample Groups**

In [ ]:
adata.obs["cell_type"].value_counts(dropna=False)

In [ ]:
color_dict = {
    'LuminalA Cancer Cells': '#ed6b9f',   
    'LuminalB Cancer Cells': '#d83890',
    'HER2E Cancer Cells': '#aa1a7d',
    'Basal Cancer Cells': '#cb2426',
    'Basal Myoepithelial Cells': '#7CBB5F',
    'Mature Luminal Cells': '#9DC3C3',          # Luminal BC 起源
    'Luminal Progenitor Cells': '#D3EDA8',      # Basal BC 起源
    "Cycling T": '#443c8c',                     # '#fff9a1', 
    "Cycling PVL Cells": '#de933e',             # '#ffda8e',
    "Cycling Myeloid Cells": '#527d48',         # '#ffbb7c', 
    "Cycling Epithelial Cells": '#8c1877',      # '#ff9c6a',
    "Tip (RGS5)": '#dbcec3', 
    "Tip (CXCL12)": '#ad9f93', 
    "Stalk": '#918479', 
    "Lymphatic": '#75695e',
    "Normal Fibroblasts": "#ffd4af", 
    "Myofibroblastic CAF (myCAF)": '#e3b68f', 
    "Inflammatory CAF (iCAF)": '#f6be2a', 
    "Antigen-presenting CAF (apCAF)": "#cc9c1f",
    "Vascular CAF (vCAF)": "#b58c69",
    "Differentiated PVL (dPVL)": '#cf7a42', 
    "Immature-like PVL (imPVL)": '#b56635',
    "cDC1": '#a2d59b', 
    "cDC2": '#76c277', 
    "mDC": '#3bab5a',
    "pDC": '#E95351', 
    "Mast Cells": '#A89C92',
    "Neutrophil": '#555657', 
    "Monocytes (Classical)": '#BDA9E8', 
    "Monocytes (Inflammatory)": '#7961AD',
    "Macrophages (M1)": '#148843', 
    "Macrophages (M2)": '#035830', 
    "Macrophages (LAM1)": '#91C79A', 
    "Macrophages (LAM2)": '#4EA660',
    "CD4+ T": '#68559d',
    "CD8+ T": '#9f9ac4', 
    "Treg": '#2573b4',
    "NK": '#7f7cb6',
    "NKT": '#bebed8', 
    "B Naive": '#6baed5',
    "B Memory": '#02779b',
    "Plasma": '#9bc9dd',
    "Cycling Cells": '#BEBEBE', 
    "Endothelial Cells": '#A8A8A8', 
    "Myeloid Cells": '#8C8C8C',
    "Immune Cells (Myeloid)": '#8C8C8C', 
    "Mesenchymal Cells": '#696969', 
    "Immune Cells": '#505050',
    "Epithelial Cells": '#1A1A1A'}

In [ ]:
adata_mali = adata[adata.obs['IHC_type'].isin(['TNBC', 'ER+', 'HER2+', 'Normal'])].copy()
adata_norm = adata[adata.obs['IHC_type'] == 'Normal'].copy()

tbnk_list = ['CD4+ T', 'CD8+ T', 'Treg', 'B Naive', 'B Memory', 'NK', 'NKT', 'Plasma']

myeloid_list = ['Macrophages (LAM1)', 'Macrophages (LAM2)', 'Monocytes (Inflammatory)', 'Monocytes (Classical)', 
                'Macrophages (M2)', 'Macrophages (M1)', 'Neutrophil', 'mDC', 'cDC2', 'pDC', 'cDC1', 'Cycling Myeloid Cells', 'Mast Cells']

immune_list = tbnk_list + myeloid_list

mesandendo_list = ['Normal Fibroblasts', 'Myofibroblastic CAF (myCAF)', 'Antigen-presenting CAF (apCAF)', 'Vascular CAF (vCAF)', 'Inflammatory CAF (iCAF)', 'Immature-like PVL (imPVL)', 'Differentiated PVL (dPVL)', 'Stalk', 'Tip (CXCL12)', 'Tip (RGS5)', 'Lymphatic']

epithelial_list = ['LuminalB Cancer Cells', 'Mature Luminal Cells', 'LuminalA Cancer Cells', 
                   'Basal Cancer Cells', 'HER2E Cancer Cells', 'Luminal Progenitor Cells', 
                   'Cycling Epithelial Cells', 'Basal Myoepithelial Cells']

In [ ]:
def cell_prop_bars(tmp_adata_mali):
    tmp = tmp_adata_mali.obs.groupby(['IHC_type', 'cell_type']).size().unstack(fill_value=0)
    tmp_pct = tmp.div(tmp.sum(axis=1), axis=0) * 100

    ordered_columns = [col for col in color_dict.keys() if col in tmp_pct.columns]
    extra_columns = [col for col in tmp_pct.columns if col not in color_dict.keys()]
    if extra_columns:
        ordered_columns.extend(extra_columns)

    tmp_pct_ordered = tmp_pct[ordered_columns]

    fig, ax = plt.subplots(figsize=(16, 5))

    bars = tmp_pct_ordered.plot(kind='barh', stacked=True, ax=ax, color=[color_dict[col] for col in tmp_pct_ordered.columns], legend=False, edgecolor='white', linewidth=0.5)

    ax.set_xlabel('Proportion (%)', fontsize=12)
    ax.set_ylabel('IHC Type', fontsize=12)
    ax.set_title('Cell Type Composition by IHC Type', fontsize=14)

    ax.grid(axis='x', alpha=0.3, linestyle='--')

    for bar in ax.patches:
        width = bar.get_width()
        if width > 3: 
            ax.annotate(f'{width:.1f}%', 
                        (bar.get_x() + width/2, bar.get_y() + bar.get_height()/2),
                        ha='center', va='center', fontsize=8, color='white', fontweight='bold')

    handles, labels = ax.get_legend_handles_labels()
    handles = handles[::-1]
    labels = labels[::-1]

    legend = ax.legend(handles, labels, title='Cell Types', bbox_to_anchor=(1.02, 1), loc='upper left',
                       frameon=True, fancybox=True, shadow=False, fontsize=9, title_fontsize=10)

    plt.tight_layout()
    plt.subplots_adjust(right=0.78)

    plt.show()

In [ ]:
tmp_adata_mali = adata_mali[adata_mali.obs['cell_type'].isin(tbnk_list)].copy()
cell_prop_bars(tmp_adata_mali)

In [ ]:
tmp_adata_mali = adata_mali[adata_mali.obs['cell_type'].isin(myeloid_list)].copy()
cell_prop_bars(tmp_adata_mali)

In [ ]:
tmp_adata_mali = adata_mali[adata_mali.obs['cell_type'].isin(mesandendo_list)].copy()
cell_prop_bars(tmp_adata_mali)

In [ ]:
tmp_adata_mali = adata_mali[adata_mali.obs['cell_type'].isin(epithelial_list)].copy()
cell_prop_bars(tmp_adata_mali)

In [ ]:
# 样本量太少了 应该做 mannwhitneyu(g1, g2, alternative='two-sided')
tmp_adata_mali = adata_mali[adata_mali.obs['cell_type'].isin(immune_list)].copy()
IHC_adata = tmp_adata_mali[tmp_adata_mali.obs["IHC_type"].isin(['ER+', 'HER2+', 'TNBC'])].copy()

IHC_2type_map = {'ER+': 'ER+/HER2+', 'HER2+': 'ER+/HER2+', 'TNBC': 'TNBC'}
IHC_adata.obs['IHC_2type'] = IHC_adata.obs['IHC_type'].astype(str).map(IHC_2type_map).astype('category')

count_df = IHC_adata.obs.groupby(['batch', 'IHC_2type', 'cell_type'], observed=True).size().unstack(fill_value=0)
prop_df = count_df.div(count_df.sum(axis=1), axis=0).reset_index()

# target_cell_list = [c for c in IHC_adata.obs['cell_type'].unique() if c in prop_df.columns]
target_cell_list = ['Neutrophil', 'CD8+ T', 'Macrophages (LAM2)', 'mDC', 'Monocytes (Inflammatory)']
n_cells = len(target_cell_list)
n_cols = 5
n_rows = math.ceil(n_cells / n_cols)

sns.set_style("ticks")
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*6, n_rows*7))
axes = axes.flatten()

existing_order = ["TNBC", "ER+/HER2+"]

for i, tgt_cell in enumerate(target_cell_list):
    ax = axes[i]
    sns.boxplot(data=prop_df, x='IHC_2type', y=tgt_cell, order=existing_order, color='white', linewidth=1.5, fliersize=0, ax=ax)
    sns.stripplot(data=prop_df, x='IHC_2type', y=tgt_cell, order=existing_order, color=".3", size=5, jitter=True, alpha=0.7, ax=ax)
    ax.set_title(tgt_cell, fontsize=14)
    ax.set_xlabel('')
    ax.set_ylabel('Proportion')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### **06. Differential Gene Analysis (and GSEA + GO/KEGG Anal as a try)**

In [ ]:
immune_adata = IHC_adata[IHC_adata.obs['cell_type'].isin(immune_list)].copy()

immune_adata = IHC_adata[IHC_adata.obs['cell_type']=='Monocytes (Inflammatory)'].copy()

In [ ]:
immune_adata.obs['IHC_2type'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(immune_adata, groupby='IHC_2type', groups=['TNBC'], reference='ER+/HER2+', method='wilcoxon')

In [ ]:
if immune_adata.X.max() > 50:
    print("not logarithmized")
else:
    print("logarithmized")

In [ ]:
deg = sc.get.rank_genes_groups_df(immune_adata, group="TNBC")

deg = deg[~deg['names'].str.startswith(('RPS', 'RPL'))]
deg = deg[~deg['names'].str.startswith('MT-')]
deg = deg[~deg['names'].str.startswith(('IGH', 'IGK', 'IGL'))]
deg['ranking'] = (deg['logfoldchanges'] * -np.log10(deg['pvals_adj'] + 1e-300))

deg.head()

In [ ]:
# sc.pl.rank_genes_groups(immune_adata, n_genes=25, sharey=False)

In [ ]:
ranked_df = deg[['names', 'logfoldchanges']].dropna()
ranked_df = ranked_df.rename(columns={'names': 'Gene', 'logfoldchanges': 'Score'})

ranked_df = deg[['names', 'ranking']]
ranked_df.columns = ['Gene', 'Score']
ranked_df.to_csv('ranked_genes.rnk', sep='\t', index=False, header=False)

pre_res = gp.prerank(
    rnk='ranked_genes.rnk',
    gene_sets='/Users/ekeulseuji/Downloads/h.all.v2026.1.Hs.symbols.gmt',
    processes=4,
    permutation_num=1000,
    outdir='gsea_results',
    seed=6,
    format='png')

res = pre_res.res2d.sort_values('NES', ascending=False)

print(res.head(5))

In [ ]:
deg = sc.get.rank_genes_groups_df(immune_adata, group="TNBC")

deg = deg[~deg['names'].str.startswith(('RPS', 'RPL'))]
deg = deg[~deg['names'].str.startswith('MT-')]
deg = deg[~deg['names'].str.startswith(('IGH', 'IGK', 'IGL'))]
remove_genes = ['MALAT1', 'TMSB4X', 'B2M', 'FTL', 'FTH1', 'EEF1A1', 'TPT1', 'ACTB', 'GAPDH']
deg = deg[~deg['names'].isin(remove_genes)] # housekeeping, ...

result_df = deg
result_df.columns = ['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']

plt.figure(figsize=(8, 6))

fold_change_threshold = 1
p_threshold = 0.05

result_df['reg'] = 'Normal'
result_df.loc[(result_df['logfoldchanges'] > fold_change_threshold) & (result_df['pvals_adj'] < p_threshold), 'reg'] = 'Up'
result_df.loc[(result_df['logfoldchanges'] < -fold_change_threshold) & (result_df['pvals_adj'] < p_threshold), 'reg'] = 'Down'

result_df['neg_log10_pval'] = -np.log10(result_df['pvals_adj'] + 1e-300)

scatter = sns.scatterplot(data=result_df, x='logfoldchanges', y='neg_log10_pval', hue='reg', 
                          palette={'Up': '#e41a1c', 'Down': '#377eb8', 'Normal': '#bdbdbd'}, edgecolor=None, s=15, alpha=0.6)

plt.axvline(x=fold_change_threshold, color='black', linestyle='--', lw=1, alpha=0.7)
plt.axvline(x=-fold_change_threshold, color='black', linestyle='--', lw=1, alpha=0.7)
plt.axhline(y=-np.log10(p_threshold), color='black', linestyle='--', lw=1, alpha=0.7)
plt.xlabel('Log2 Fold Change (TNBC vs ER+/HER2+)', fontsize=12)
plt.ylabel('-Log10 Adjusted P-value', fontsize=12)
plt.title('TNBC vs ER+/HER2+', fontsize=14)
plt.legend(title='Regulation', loc='upper left')

top_genes = result_df.nlargest(30, 'logfoldchanges')['names'].tolist()
for gene in top_genes:
    gene_data = result_df[result_df['names'] == gene].iloc[0]
    plt.annotate(gene, 
                 xy=(gene_data['logfoldchanges'], gene_data['neg_log10_pval']),
                 xytext=(5, 5), 
                 textcoords='offset points',
                 fontsize=8,
                 alpha=0.7)

plt.tight_layout()
plt.show()

print(f"Up-regulated genes: {len(result_df[result_df['reg'] == 'Up'])}")
print(f"Down-regulated genes: {len(result_df[result_df['reg'] == 'Down'])}")

In [ ]:
up_genes = result_df[(result_df['logfoldchanges'] > 1) & (result_df['pvals_adj'] < 0.05)]['names'].tolist()

enr = gp.enrichr(gene_list=up_genes,
                 gene_sets=['GO_Biological_Process_2023', 'KEGG_2021_Human'],
                 organism='human', 
                 outdir=None)

gp.barplot(enr.results, column="Adjusted P-value", group='Gene_set', size=10, top_term=10)

In [ ]:
top_genes = result_df.sort_values('logfoldchanges', ascending=False)['names'][:10].tolist() + \
            result_df.sort_values('logfoldchanges', ascending=True)['names'][:10].tolist()

sc.pl.dotplot(IHC_adata, var_names=top_genes, groupby='IHC_2type', standard_scale='var', cmap='coolwarm')

In [ ]:
ifn_genes = ['STAT1', 'IRF1', 'ISG15', 'IFIT3', 'MX1', 'IFI6', 'GBP1', 'PSMB9']
sc.tl.score_genes(immune_adata, gene_list=ifn_genes, score_name='IFN_score')

nfkb_genes = ['IL1B', 'TNFAIP3', 'NFKBIA', 'CXCL10', 'CCL5', 'IRF1', 'SAT1']
sc.tl.score_genes(immune_adata, gene_list=nfkb_genes, score_name='NFkB_score')

oxphos_genes = ['COX4I1', 'COX5B', 'NDUFA1', 'ATP5F1A', 'UQCRB', 'COX7C', 'ATP5MC2']
sc.tl.score_genes(immune_adata, gene_list=oxphos_genes, score_name='OXPHOS_score')

tgfb_genes = ['TGFB1', 'JUNB', 'ID2', 'KLF6', 'CD44', 'TGIF1']
sc.tl.score_genes(immune_adata, gene_list=tgfb_genes, score_name='TGFb_score')

score_keys = ['IFN_score', 'NFkB_score', 'OXPHOS_score', 'TGFb_score']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, key in enumerate(score_keys):
    sc.pl.violin(immune_adata, keys=key, groupby='IHC_2type', stripplot=False, ax=axes[i], show=False)
    axes[i].set_title(key)
    axes[i].set_xlabel('')
plt.tight_layout()
plt.show()

### **07. CellChat and Monocyte Trajectory**

In [ ]:
import statsmodels.compat.pandas as sm_compat
if not hasattr(sm_compat, 'PD_LT_3'):
    sm_compat.PD_LT_3 = pd.__version__ < "3.0.0"

In [ ]:
import liana as li
# sc.settings.set_figure_params(dpi=75, facecolor="white")
# import all individual methods
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean
from plotnine import theme, element_text

In [ ]:
types_to_keep = ['Basal Cancer Cells', 'HER2E Cancer Cells', 'LuminalB Cancer Cells', 'LuminalA Cancer Cells',
                 'Basal Myoepithelial Cells', 'Mature Luminal Cells', 'Luminal Progenitor Cells',
                 'Macrophages (LAM1)', 'Macrophages (LAM2)', 'Monocytes (Inflammatory)', 'Monocytes (Classical)', 'Neutrophil', 
                 'Macrophages (M2)', 'Macrophages (M1)', 'CD4+ T', 'CD8+ T', 'NK', 'NKT']

cc_type_map = {
    'LuminalA Cancer Cells': 'Malignant Epithelial', 'LuminalB Cancer Cells': 'Malignant Epithelial',
    'HER2E Cancer Cells': 'Malignant Epithelial', 'Basal Cancer Cells': 'Malignant Epithelial',
    'Basal Myoepithelial Cells': 'Normal Epithelial', 'Mature Luminal Cells': 'Normal Epithelial',
    'Luminal Progenitor Cells': 'Normal Epithelial', 'Macrophages (LAM1)': 'Macrophages (LAM1)',
    'Macrophages (LAM2)': 'Macrophages (LAM2)', 'Monocytes (Inflammatory)': 'Monocytes (Inflammatory)',
    'Monocytes (Classical)': 'Monocytes (Classical)', 'Neutrophil': 'Neutrophil', 'Macrophages (M2)': 'Macrophages (M2)',
    'Macrophages (M1)': 'Macrophages (M1)', 'CD4+ T': 'CD4+ T', 'CD8+ T': 'CD8+ T', 'NK': 'NK', 'NKT': 'NKT'}

adata_nTN = adata[adata.obs['IHC_type'].isin(['ER+', 'HER2+'])].copy()
adata_TN = adata[adata.obs['IHC_type'] == 'TNBC'].copy()

adata_nTN = adata_nTN[adata_nTN.obs['cell_type'].isin(types_to_keep)].copy()
adata_TN = adata_TN[adata_TN.obs['cell_type'].isin(types_to_keep)].copy()

adata_nTN.obs['cc_cell_type'] = adata_nTN.obs['cell_type'].astype(str).map(cc_type_map).astype('category')
# print(adata_nTN.obs['cc_cell_type'].value_counts(dropna=False))
adata_TN.obs['cc_cell_type'] = adata_TN.obs['cell_type'].astype(str).map(cc_type_map).astype('category')
# print(adata_TN.obs['cc_cell_type'].value_counts(dropna=False))

In [ ]:
li.mt.cellchat(adata_nTN, groupby='cc_cell_type',
               resource_name='cellchatdb',
               expr_prop=0.1, min_cells=5, inplace=True)

li.mt.cellchat(adata_TN, groupby='cc_cell_type',
               resource_name='cellchatdb',
               expr_prop=0.1, min_cells=5, inplace=True)

res_nTN = adata_nTN.uns['liana_res']
res_nTN['scaled_weight'] = res_nTN['lr_probs'] ** 10
adata_nTN.uns['liana_res'] = res_nTN

res_TN = adata_TN.uns['liana_res']
res_TN['scaled_weight'] = res_TN['lr_probs'] ** 10
adata_TN.uns['liana_res'] = res_TN

p = li.pl.dotplot(adata = adata_nTN, 
              colour='lr_probs',
              size='cellchat_pvals',
              inverse_size=True, # we inverse sign since we want small p-values to have large sizes
              source_labels=['Malignant Epithelial', 'Normal Epithelial'],
              target_labels=['Macrophages (LAM1)', 'Macrophages (LAM2)', 'Monocytes (Inflammatory)', 'Monocytes (Classical)', 'Neutrophil', 
                 'Macrophages (M2)', 'Macrophages (M1)', 'CD4+ T', 'CD8+ T', 'NK', 'NKT'],
              figure_size=(11, 7),
              # finally, since cpdbv2 suggests using a filter to FPs
              # we filter the pvals column to <= 0.05
              filter_fun=lambda x: x['cellchat_pvals'] <= 0.05,
              uns_key='liana_res')

p = p + theme(axis_text_x=element_text(rotation=45, hjust=1))
p

In [ ]:
liana_res = adata_TN.uns['liana_res']
unique_sources = liana_res['source'].unique()
# print(unique_sources)

p = li.pl.dotplot(adata = adata_TN, 
              colour='lr_probs',
              size='cellchat_pvals',
              inverse_size=True, # we inverse sign since we want small p-values to have large sizes
              source_labels=['Malignant Epithelial', 'Normal Epithelial'],
              target_labels=['Macrophages (LAM1)', 'Macrophages (LAM2)', 'Monocytes (Inflammatory)', 'Monocytes (Classical)', 'Neutrophil', 
                 'Macrophages (M2)', 'Macrophages (M1)', 'CD4+ T', 'CD8+ T', 'NK', 'NKT'],
              figure_size=(11, 7),
              # finally, since cpdbv2 suggests using a filter to FPs
              # we filter the pvals column to <= 0.05
              filter_fun=lambda x: x['cellchat_pvals'] <= 0.05,
              uns_key='liana_res')

p = p + theme(axis_text_x=element_text(rotation=45, hjust=1))
p

In [ ]:
li.pl.circle_plot(
    adata_nTN,
    groupby='cc_cell_type',
    score_key='scaled_weight',
    inverse_score=True,
    # target_labels=['CD8+ T Cells', 'NK Cells'],
    filter_fun=lambda x: x['cellchat_pvals'] <= 0.05,
    figure_size=(10, 10))

In [ ]:
li.pl.circle_plot(
    adata_TN,
    groupby='cc_cell_type',
    score_key='scaled_weight',
    inverse_score=True,
    # target_labels=['CD8+ T Cells', 'NK Cells'],
    filter_fun=lambda x: x['cellchat_pvals'] <= 0.05,
    figure_size=(10, 10))

In [ ]:
res_TN['group'] = 'TNBC'
res_nTN['group'] = 'ER+/HER2+'
res_All = pd.concat([res_TN, res_nTN], ignore_index=True)

# 按ligand-receptor对去比较平均 scaled_weight
pivot = res_All.pivot_table(index=['ligand_complex', 'receptor_complex'], 
                            columns='group', values='scaled_weight', aggfunc='mean')

pivot['log2FC'] = np.log2(pivot['TNBC'] + 0.01) - np.log2(pivot['ER+/HER2+'] + 0.01)

pivot.sort_values('log2FC', ascending=False).head(15)  # TNBC中显著增强的LR对

In [ ]:
p = li.pl.tileplot(
    adata = adata_nTN, 
    fill='trimean',
    label='props',
    label_fun=lambda x: f'{x:.2f}',
    top_n=15, 
    orderby='cellchat_pvals',
    orderby_ascending=True,
    uns_key='liana_res',
    source_labels=['Malignant Epithelial', 'Normal Epithelial'],
    target_labels=['Macrophages (LAM1)', 'Macrophages (LAM2)', 'Monocytes (Inflammatory)', 'Monocytes (Classical)', 'Neutrophil', 
                 'Macrophages (M2)', 'Macrophages (M1)', 'CD4+ T', 'CD8+ T', 'NK', 'NKT'],
    source_title='Ligand',
    target_title='Receptor',
    figure_size=(12, 6))

p = p + theme(axis_text_x=element_text(rotation=45, hjust=1))
p

In [ ]:
p = li.pl.tileplot(
    adata = adata_TN, 
    fill='trimean',
    label='props',
    label_fun=lambda x: f'{x:.2f}',
    top_n=15, 
    orderby='cellchat_pvals',
    orderby_ascending=True,
    uns_key='liana_res',
    source_labels=['Malignant Epithelial', 'Normal Epithelial'],
    target_labels=['Macrophages (LAM1)', 'Macrophages (LAM2)', 'Monocytes (Inflammatory)', 'Monocytes (Classical)', 'Neutrophil', 
                 'Macrophages (M2)', 'Macrophages (M1)', 'CD4+ T', 'CD8+ T', 'NK', 'NKT'],
    source_title='Ligand',
    target_title='Receptor',
    figure_size=(12, 6))

p = p + theme(axis_text_x=element_text(rotation=45, hjust=1))
p

In [ ]:
def mono_traj(adata_temp, cell_type):
    sc.pp.highly_variable_genes(adata_temp, n_top_genes=2000)
    sc.pp.filter_genes(adata_temp, min_cells=8)
    sc.pp.scale(adata_temp, max_value=10)
    sc.tl.pca(adata_temp, svd_solver='arpack')
    sc.pp.neighbors(adata_temp, n_pcs=20)
    sc.tl.leiden(adata_temp, resolution=0.8)
    sc.tl.umap(adata_temp)
    
    print(cell_type)
    umap = adata_temp.obsm['X_umap']; leiden = adata_temp.obs['leiden']; IHC_2type = adata_temp.obs['IHC_2type']
    leiden_numeric = leiden.cat.codes.values
    IHC_2type_numeric = IHC_2type.cat.codes.values
    print(f"UMAP shape: {umap.shape}")
    print(f"Leiden unique values: {np.unique(leiden_numeric)}")
    print(f"Number of clusters: {len(np.unique(leiden_numeric))}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    scatter1 = ax1.scatter(umap[:, 0], umap[:, 1], c=leiden_numeric, s=12, cmap='Paired', alpha=0.8)
    ax1.set_title(f'{cell_type} - Leiden Clusters')
    ax1.set_xlabel('UMAP 1')
    ax1.set_ylabel('UMAP 2')

    categories = adata_temp.obs['IHC_2type'].cat.categories
    IHC_2type_numeric = adata_temp.obs['IHC_2type'].cat.codes
    colors = ['#216b8a', '#d96143']

    for i, cat in enumerate(categories):
        mask = IHC_2type_numeric == i
        ax2.scatter(umap[mask, 0], umap[mask, 1], s=12, alpha=0.8, label=cat, color=colors[i % len(colors)])

    ax2.set_title(f'{cell_type} - IHC Types')
    ax2.set_xlabel('UMAP 1')
    ax2.set_ylabel('UMAP 2')

    ax2.legend(title='IHC Type', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()


    sc.pp.highly_variable_genes(adata_temp, n_top_genes=2000)
    sc.pp.filter_genes(adata_temp, min_cells=8)
    sc.pp.scale(adata_temp, max_value=10)
    sc.tl.pca(adata_temp, svd_solver='arpack')
    sc.pp.neighbors(adata_temp, n_pcs=20)
    sc.tl.leiden(adata_temp, resolution=0.5)
    sc.tl.umap(adata_temp)
    
    umap = adata_temp.obsm['X_umap']; leiden = adata_temp.obs['leiden']; IHC_2type = adata_temp.obs['IHC_2type']
    leiden_numeric = leiden.cat.codes.values
    IHC_2type_numeric = IHC_2type.cat.codes.values
    print(f"UMAP shape: {umap.shape}")
    print(f"Leiden unique values: {np.unique(leiden_numeric)}")
    print(f"Number of clusters: {len(np.unique(leiden_numeric))}")
    
    adata_temp_TN = adata_temp[adata_temp.obs['IHC_type'].isin(['TNBC'])].copy()
    adata_temp_nTN = adata_temp[adata_temp.obs['IHC_type'].isin(['ER+', 'HER2+'])].copy()
    
    umap_TN = adata_temp_TN.obsm['X_umap']; leiden_TN = adata_temp_TN.obs['leiden']
    umap_nTN = adata_temp_nTN.obsm['X_umap']; leiden_nTN = adata_temp_nTN.obs['leiden']

    leiden_numeric_TN = leiden_TN.cat.codes.values
    leiden_numeric_nTN = leiden_nTN.cat.codes.values
    # leiden_numeric = leiden.astype(int).values

    expression_matrix_TN = adata_temp_TN.X
    expression_matrix_nTN = adata_temp_nTN.X

    barcodes_TN = adata_temp_TN.obs_names.tolist(); features_TN = adata_temp_TN.var_names.tolist()
    barcodes_nTN = adata_temp_nTN.obs_names.tolist(); features_nTN = adata_temp_nTN.var_names.tolist()

    projected_points_TN, mst_TN, centroids_TN = learn_graph(matrix=umap_TN, clusters=leiden_numeric_TN)
    projected_points_nTN, mst_nTN, centroids_nTN = learn_graph(matrix=umap_nTN, clusters=leiden_numeric_nTN)

    root_cell_index = 0 
    pseudotime_TN = order_cells(
        umap_TN, centroids_TN,
        mst=mst_TN,
        projected_points=projected_points_TN,
        root_cells=root_cell_index,)

    print(f"TNBC Pseudotime computed: min={pseudotime_TN.min():.2f}, max={pseudotime_TN.max():.2f}")
    
    pseudotime_nTN = order_cells(
        umap_nTN, centroids_nTN,
        mst=mst_nTN,
        projected_points=projected_points_nTN,
        root_cells=root_cell_index,)

    print(f"ER+/HER2+ Pseudotime computed: min={pseudotime_nTN.min():.2f}, max={pseudotime_nTN.max():.2f}")
    
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.5, 4))

    scatter1 = ax1.scatter(umap_TN[:, 0], umap_TN[:, 1], c=leiden_numeric_TN, s=12, cmap="Paired")
    ax1.set_title(f"Leiden Clusters (TNBC)", fontsize=14)
    ax1.set_xticks([])
    ax1.set_yticks([])

    edges = np.array(mst_TN.nonzero()).T
    for edge in edges:
        ax1.plot(centroids_TN[edge, 0], centroids_TN[edge, 1], c="black", linewidth=1)

    # cbar1 = plt.colorbar(scatter1, ax=ax1, label='Leiden cluster', fraction=0.046, pad=0.04)

    scatter2 = ax2.scatter(umap_TN[:, 0], umap_TN[:, 1], c=pseudotime_TN, s=12, cmap="viridis", alpha=0.8)
    ax2.set_title(f"Pseudo-time Trajectory (TNBC)", fontsize=14)
    ax2.set_xticks([])
    ax2.set_yticks([])

    for edge in edges:
        ax2.plot(centroids_TN[edge, 0], centroids_TN[edge, 1], c="black", linewidth=1)

    cbar2 = plt.colorbar(scatter2, ax=ax2, label='Pseudotime', fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
    

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.5, 4))

    scatter1 = ax1.scatter(umap_nTN[:, 0], umap_nTN[:, 1], c=leiden_numeric_nTN, s=12, cmap="Paired")
    ax1.set_title(f"Leiden Clusters (ER+/HER2+)", fontsize=14)
    ax1.set_xticks([])
    ax1.set_yticks([])

    edges = np.array(mst_nTN.nonzero()).T
    for edge in edges:
        ax1.plot(centroids_nTN[edge, 0], centroids_nTN[edge, 1], c="black", linewidth=1)

    # cbar1 = plt.colorbar(scatter1, ax=ax1, label='Leiden cluster', fraction=0.046, pad=0.04)

    scatter2 = ax2.scatter(umap_nTN[:, 0], umap_nTN[:, 1], c=pseudotime_nTN, s=12, cmap="viridis", alpha=0.8)
    ax2.set_title(f"Pseudo-time Trajectory (ER+/HER2+)", fontsize=14)
    ax2.set_xticks([])
    ax2.set_yticks([])

    for edge in edges:
        ax2.plot(centroids_nTN[edge, 0], centroids_nTN[edge, 1], c="black", linewidth=1)

    cbar2 = plt.colorbar(scatter2, ax=ax2, label='Pseudotime', fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [ ]:
adata_temp = adata[adata.obs['IHC_type'].isin(['ER+', 'HER2+', 'TNBC'])].copy()
adata_temp.obs['IHC_2type'] = adata_temp.obs['IHC_type'].astype(str).map(IHC_2type_map).astype('category')
adata_temp = adata_temp[adata_temp.obs['cell_type']=='Monocytes (Inflammatory)'].copy()
mono_traj(adata_temp, "Monocytes (Inflammatory)")

In [ ]:
morans_i_scores_TN, pvalues_TN, adjusted_pvalues_TN = differential_expression_genes(expression_matrix_TN,
                                                                                    projected_cells=projected_points_TN)
    
morani_results_TN = pd.DataFrame({
    "Moran's I score": morans_i_scores_TN,
    "P-values": np.round(pvalues_TN, 4),
    "Adjusted P-values": np.round(adjusted_pvalues_TN, 4),
    "Gene Indices": np.arange(len(morans_i_scores_TN))},
    index=pd.Series(features_TN, name="Genes"),).sort_values("Moran's I score", ascending=False)
    
print(morani_results_TN.head(15))
    
morans_i_scores_nTN, pvalues_nTN, adjusted_pvalues_nTN = differential_expression_genes(expression_matrix_nTN,
                                                                                       projected_cells=projected_points_nTN)
morani_results_nTN = pd.DataFrame({
    "Moran's I score": morans_i_scores_nTN,
    "P-values": np.round(pvalues_nTN, 4),
    "Adjusted P-values": np.round(adjusted_pvalues_nTN, 4),
    "Gene Indices": np.arange(len(morans_i_scores_nTN))},
    index=pd.Series(features_nTN, name="Genes"),).sort_values("Moran's I score", ascending=False)
    
print(morani_results_nTN.head(15))

In [ ]:
plt.figure(figsize=(16, 10))

for i, (gene_name, row) in enumerate(morani_results_TN.head(12).iterrows()):
    gene_idx = int(row["Gene Indices"])
    
    plt.subplot(3, 4, i + 1)
    expr = expression_matrix_TN[:, gene_idx]
    if hasattr(expr, "toarray"):
        expr = expr.toarray().flatten()
        
    plt.scatter(umap_TN[:, 0], umap_TN[:, 1], 
                c=expr,
                s=8, cmap="Reds")
    plt.xticks([])
    plt.yticks([])
    plt.title(gene_name)

plt.tight_layout()
plt.show()

In [ ]:
sc.tl.paga(adata_temp, groups='leiden')

# sc.pl.paga(adata_temp, color='leiden', title='PAGA Graph')
# sc.tl.draw_graph(adata_temp, init_pos='paga')

# DPT (Diffusion Pseudotime)
adata_temp.uns['iroot'] = np.flatnonzero(adata_temp.obs['leiden'] == '0')[0]
sc.tl.dpt(adata_temp)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.paga(adata_temp, color='leiden', show=False, ax=axes[0])
axes[0].set_title('PAGA: Cluster Connectivity')

sc.pl.draw_graph(adata_temp, color='dpt_pseudotime', 
                 title='DPT Pseudotime Trajectory Tree', 
                 show=False, ax=axes[1], legend_loc='on data')

plt.tight_layout()
plt.show()

### **08. TNBC-Specific TFs and the Target Genes' GO/KEGG Enrichment Analysis**

In [ ]:
dorothea_relations = dc.op.dorothea(organism='human')
print(dorothea_relations.head())
print('')

tf_list = dorothea_relations['source'].unique()
print(f"Total n of TFs: {len(tf_list)}")
print(tf_list[:20])

In [ ]:
tn_top_genes = morani_results_TN.head(200).index.tolist()
ntn_top_genes = morani_results_nTN.head(200).index.tolist()

tn_specific = set(tn_top_genes) - set(ntn_top_genes)

tn_specific_by_ratio = [
    g for g in tn_top_genes 
    if morani_results_TN.loc[g, "Moran's I score"] / 
       (morani_results_nTN.loc[g, "Moran's I score"] + 0.01) > 2]

In [ ]:
# TNBC特异性且随轨迹波动的 TF
tn_tf_candidates = [g for g in tn_specific_by_ratio if g in tf_list]

print(f"{len(tn_tf_candidates)} specific TFs: {tn_tf_candidates[:5]}")

In [ ]:
def get_lightweight_regulon(tf, expression_matrix, threshold=0.5):
    tf_expr = expression_matrix[tf].values
    corrs = []
    for gene in expression_matrix.columns:
        if gene == tf: continue
        r, p = spearmanr(tf_expr, expression_matrix[gene].values)
        corrs.append((gene, r, p))
    df_corr = pd.DataFrame(corrs, columns=['Target', 'r', 'p'])
    regulon = df_corr[(df_corr['r'] > threshold) & (df_corr['p'] < 0.05)]
    return regulon.sort_values('r', ascending=False)

def get_regulon(tf, df_expression, threshold):
    tf_expr = df_expression[tf].values
    corrs = [] # 算所有基因与该TF的Spearman相关性
    for gene in df_expression.columns:
        if gene == tf: continue
        r, p = spearmanr(tf_expr, df_expression[gene].values)
        corrs.append((gene, r, p))

    df_corr = pd.DataFrame(corrs, columns=['Target', 'r', 'p'])
    regulon = df_corr[(df_corr['r'].abs() > threshold) & (df_corr['p'] < 0.05)] # 筛选高相关且显著的基因 (r高则强正相关)
    return regulon.sort_values('r', ascending=False)

In [ ]:
df_expr_TN = pd.DataFrame(
    expression_matrix_TN.toarray() if hasattr(expression_matrix_TN, "toarray") else expression_matrix_TN,
    columns=features_TN,
    index=barcodes_TN)

df_expr_nTN = pd.DataFrame(
    expression_matrix_nTN.toarray() if hasattr(expression_matrix_nTN, "toarray") else expression_matrix_nTN,
    columns=features_nTN,
    index=barcodes_nTN)

In [ ]:
temp_regulon = get_regulon("CREM", df_expr_TN, 0.15)
print(f"Number of Target Gene(s): {len(temp_regulon)}") # 候选靶基因数量
print(temp_regulon.head())

temp_targets = temp_regulon['Target'].tolist()
if len(temp_targets) > 0:    
    sc.tl.score_genes(adata_temp_TN, gene_list=temp_targets, score_name='CREM_regulon_score')
    plot_df = pd.DataFrame({
        'Pseudotime': pseudotime_TN,
        'CREM_Activity': adata_temp_TN.obs['CREM_regulon_score'],
        'Group': 'TNBC'})
    plt.figure(figsize=(6, 4))
    sns.lineplot(data=plot_df, x='Pseudotime', y='CREM_Activity', color='#d96143')
    plt.title("CREM Regulon Activity along Pseudotime (TNBC)")
    plt.xlabel("DPT Pseudotime")
    plt.ylabel("Regulatory Activity Score")
    plt.show()
else:
    print("No Target Gene Detected")

target_genes1 = temp_regulon['Target'].tolist()

In [ ]:
temp_regulon = get_regulon("CREM", df_expr_nTN, 0.15)
print(f"Number of Target Gene(s): {len(temp_regulon)}") # 候选靶基因数量
print(temp_regulon.head())

temp_targets = temp_regulon['Target'].tolist()
if len(temp_targets) > 0:    
    sc.tl.score_genes(adata_temp_nTN, gene_list=temp_targets, score_name='CREM_regulon_score')
    plot_df = pd.DataFrame({
        'Pseudotime': pseudotime_nTN,
        'CREM_Activity': adata_temp_nTN.obs['CREM_regulon_score'],
        'Group': 'non-TNBC'})
    plt.figure(figsize=(6, 4))
    sns.lineplot(data=plot_df, x='Pseudotime', y='CREM_Activity', color='#6a4394')
    plt.title("CREM Regulon Activity along Pseudotime (non-TNBC)")
    plt.xlabel("DPT Pseudotime")
    plt.ylabel("Regulatory Activity Score")
    plt.show()
else:
    print("No Target Gene Detected")

In [ ]:
temp_regulon = get_regulon("MAF", df_expr_TN, 0.15)
print(f"Number of Target Gene(s): {len(temp_regulon)}") # 候选靶基因数量
print(temp_regulon.head())
print('')

temp_targets = temp_regulon['Target'].tolist()
if len(temp_targets) > 0:
    sc.tl.score_genes(adata_temp_TN, gene_list=temp_targets, score_name='MAF_regulon_score')
    plot_df = pd.DataFrame({
        'Pseudotime': pseudotime_TN,
        'MAF_Activity': adata_temp_TN.obs['MAF_regulon_score'],
        'Group': 'TNBC'})
    plt.figure(figsize=(6, 4))
    sns.lineplot(data=plot_df, x='Pseudotime', y='MAF_Activity', color='#d96143')
    plt.title("MAF Regulon Activity along Pseudotime (TNBC)")
    plt.xlabel("DPT Pseudotime")
    plt.ylabel("Regulatory Activity Score")
    plt.show()
else:
    print("No Target Gene Detected")

target_genes2 = temp_regulon['Target'].tolist()

In [ ]:
temp_regulon = get_regulon("MAF", df_expr_nTN, 0.15)
print(f"Number of Target Gene(s): {len(temp_regulon)}") # 候选靶基因数量
print(temp_regulon.head())
print('')

temp_targets = temp_regulon['Target'].tolist()
if len(temp_targets) > 0:
    sc.tl.score_genes(adata_temp_nTN, gene_list=temp_targets, score_name='MAF_regulon_score')
    plot_df = pd.DataFrame({
        'Pseudotime': pseudotime_nTN,
        'MAF_Activity': adata_temp_nTN.obs['MAF_regulon_score'],
        'Group': 'non-TNBC'})
    plt.figure(figsize=(6, 4))
    sns.lineplot(data=plot_df, x='Pseudotime', y='MAF_Activity', color='#6a4394')
    plt.title("MAF Regulon Activity along Pseudotime (non-TNBC)")
    plt.xlabel("DPT Pseudotime")
    plt.ylabel("Regulatory Activity Score")
    plt.show()
else:
    print("No Target Gene Detected")

In [ ]:
temp_regulon = get_regulon("EOMES", df_expr_TN, 0.15)
print(f"Number of Target Gene(s): {len(temp_regulon)}") # 候选靶基因数量
print(temp_regulon.head())
print('')

temp_targets = temp_regulon['Target'].tolist()
if len(temp_targets) > 0:
    sc.tl.score_genes(adata_temp_TN, gene_list=temp_targets, score_name='EOMES_regulon_score')
    plot_df = pd.DataFrame({
        'Pseudotime': pseudotime_TN,
        'EOMES_Activity': adata_temp_TN.obs['EOMES_regulon_score'],
        'Group': 'TNBC'})
    plt.figure(figsize=(6, 4))
    sns.lineplot(data=plot_df, x='Pseudotime', y='EOMES_Activity', color='#d96143')
    plt.title("EOMES Regulon Activity along Pseudotime (TNBC)")
    plt.xlabel("DPT Pseudotime")
    plt.ylabel("Regulatory Activity Score")
    plt.show()
else:
    print("No Target Gene Detected")

target_genes3 = temp_regulon['Target'].tolist()

In [ ]:
temp_regulon = get_regulon("EOMES", df_expr_nTN, 0.15)
print(f"Number of Target Gene(s): {len(temp_regulon)}") # 候选靶基因数量
print(temp_regulon.head())
print('')

temp_targets = temp_regulon['Target'].tolist()
if len(temp_targets) > 0:
    sc.tl.score_genes(adata_temp_nTN, gene_list=temp_targets, score_name='EOMES_regulon_score')
    plot_df = pd.DataFrame({
        'Pseudotime': pseudotime_nTN,
        'EOMES_Activity': adata_temp_nTN.obs['EOMES_regulon_score'],
        'Group': 'non-TNBC'})
    plt.figure(figsize=(6, 4))
    sns.lineplot(data=plot_df, x='Pseudotime', y='EOMES_Activity', color='#6a4394')
    plt.title("EOMES Regulon Activity along Pseudotime (non-TNBC)")
    plt.xlabel("DPT Pseudotime")
    plt.ylabel("Regulatory Activity Score")
    plt.show()
else:
    print("No Target Gene Detected")

In [ ]:
target_genes = target_genes1 + target_genes2 + target_genes3

enr = gp.enrichr(
    gene_list=target_genes,
    gene_sets=['GO_Biological_Process_2023', 'KEGG_2021_Human'],
    organism='human',
    outdir=None)

gp.barplot(enr.results, column="Adjusted P-value", top_term=10)